# PhytoCYP: research pipeline

Predicted inhibition of five cytochrome P450 enzymes (CYP1A2, CYP2C9, CYP2C19, CYP2D6, CYP3A4) by compounds of 17
medicinal herbs, with checks against published laboratory measurements, a regulator's signal report and herb-guide
documentation. The notebook runs top to bottom on a fresh kernel; all results are written to `outputs/`
(`outputs/MAPPING.md` lists which output feeds which number).

Sections: 1 setup and configuration, 2 data loading and preprocessing, 3 modelling (five enzymes), 4 evaluation,
5 herb-drug pairs and checks (ChEMBL, SFDA, prior documentation), 6 herb-level scoring, 7 pharmacogenomic layer,
8 comparison with an existing tool (not a validation), 9 summary numbers, figures and tables.

## 1. Setup and configuration

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.rcParams["figure.dpi"] = 100

ROOT = Path.cwd()
assert (ROOT / "src").is_dir() and (ROOT / "data").is_dir(), "run the notebook from the repository root"
PROJECT = ROOT                      # kept so the ported cells below read naturally
DATA = ROOT / "data"
OUT = ROOT / "outputs"
RESULTS = OUT
(OUT).mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]
FLAG_THRESHOLD = 0.5                # probability at or above which a compound is called an inhibitor
CHEMBL_CUTOFF_NM = 10_000           # a measured IC50 or Ki up to 10 uM counts as an inhibitor (same criterion as the training labels)

import cyp_pipeline as cp
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Data loading and preprocessing, and 3. modelling (all five enzymes)

The five training sets are the TDC `CYP*_Veith` datasets (Veith et al. 2009, PubChem AID 1851), downloaded on first run
into `data/raw/`. Each set is cleaned (largest fragment, neutralisation, removal of conflicting labels and duplicates),
featurised (2,048 Morgan bits of radius 2 plus 10 RDKit descriptors), evaluated by scaffold-grouped 5-fold
cross-validation and by one scaffold split, and a final XGBoost model is trained on all compounds. The herb compounds are
scored with each final model and flagged as inside or outside the applicability domain.

In [ ]:
import sys

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

import cyp_pipeline as cp

print("Enzymes to model:")
for enzyme, dataset in cp.ENZYMES.items():
    print(f"  {enzyme:<9} {dataset}")

In [ ]:
from tdc.single_pred import ADME

Path("models/all_enzymes").mkdir(parents=True, exist_ok=True)
Path("data/processed/all_enzymes").mkdir(parents=True, exist_ok=True)

summary_rows, cleaning_rows = [], []
herb_models = {}

for enzyme, dataset in cp.ENZYMES.items():
    print(f"\n=== {enzyme} ===")

    raw = ADME(name=dataset, path="data/raw").get_data()
    clean, report = cp.clean_dataset(raw)
    print(f"  {report['starting_rows']} -> {report['final_rows']} compounds "
          f"({report['duplicates']} duplicates, {report['label_conflicts']} conflicts), "
          f"{report['inhibitor_fraction']*100:.1f}% inhibitors")

    X = cp.featurise(clean["SMILES_std"])
    y = clean["Y"].to_numpy()
    scaffolds = cp.scaffolds_of(clean["SMILES_std"])

    cv = cp.cross_validate(X, y, scaffolds)
    train_idx, test_idx = cp.scaffold_split(scaffolds)
    hard_model = cp.train_model(X[train_idx], y[train_idx])
    hard = cp.score(y[test_idx], hard_model.predict_proba(X[test_idx])[:, 1])

    print(f"  CV AUROC {cv['AUROC'].mean():.3f} +/- {cv['AUROC'].std():.3f}"
          f"   scaffold-split AUROC {hard['AUROC']:.3f}")

    # Train the final model on everything, and keep it
    final_model = cp.train_model(X, y)
    joblib.dump(final_model, f"models/all_enzymes/{enzyme}_xgboost.joblib")
    np.savez_compressed(f"data/processed/all_enzymes/{enzyme}_features.npz", X=X, y=y)
    clean[["Drug_ID", "SMILES_std", "Y"]].to_csv(
        f"data/processed/all_enzymes/{enzyme}_standardised.csv", index=False)
    herb_models[enzyme] = (final_model, clean["SMILES_std"].tolist())

    cleaning_rows.append({"Enzyme": enzyme, **report})
    summary_rows.append({
        "Enzyme": enzyme,
        "Compounds": report["final_rows"],
        "% inhibitors": round(report["inhibitor_fraction"] * 100, 1),
        "Unique scaffolds": len(set(scaffolds)),
        "CV AUROC": round(cv["AUROC"].mean(), 3),
        "CV AUROC sd": round(cv["AUROC"].std(), 3),
        "CV AUPRC": round(cv["AUPRC"].mean(), 3),
        "CV AUPRC sd": round(cv["AUPRC"].std(), 3),
        "CV sensitivity": round(cv["Sensitivity"].mean(), 3),
        "CV specificity": round(cv["Specificity"].mean(), 3),
        "Scaffold-split AUROC": round(hard["AUROC"], 3),
    })

print("\nAll five done.")

In [ ]:
summary = pd.DataFrame(summary_rows).set_index("Enzyme")
display(summary)

summary.to_csv("outputs/table2_model_validation_per_enzyme.csv")
pd.DataFrame(cleaning_rows).to_csv("outputs/table1_training_data_cleaning.csv", index=False)
print("Saved to outputs/table2_model_validation_per_enzyme.csv")

In [ ]:
from tdc.benchmark_group import admet_group

LEADERS = {   # top published AUPRC, retrieved 26 Oct 2026
    "CYP2C9": ("MapLight + GNN", 0.859),
    "CYP2D6": ("MapLight + GNN", 0.790),
    "CYP3A4": ("MapLight + GNN", 0.916),
}

group = admet_group(path="data/raw/")
benchmark_rows = []

for enzyme in ["CYP2C9", "CYP2D6", "CYP3A4"]:
    dataset = cp.ENZYMES[enzyme]
    predictions_list = []
    for seed in [1, 2, 3, 4, 5]:
        benchmark = group.get(dataset)
        train, valid = group.get_train_valid_split(
            benchmark=benchmark["name"], split_type="default", seed=seed)
        X_train = cp.featurise(list(train["Drug"]) + list(valid["Drug"]))
        y_train = np.concatenate([train["Y"].values, valid["Y"].values])
        model = cp.train_model(X_train, y_train, seed=seed)
        X_test = cp.featurise(list(benchmark["test"]["Drug"]))
        predictions_list.append({benchmark["name"]: model.predict_proba(X_test)[:, 1]})

    auprc, sd = group.evaluate_many(predictions_list)[dataset.lower()]
    leader_name, leader_score = LEADERS[enzyme]
    benchmark_rows.append({
        "Enzyme": enzyme,
        "Our AUPRC": round(auprc, 3),
        "sd": round(sd, 3),
        "Leaderboard best": leader_score,
        "Gap": round(auprc - leader_score, 3),
    })
    print(f"{enzyme}: {auprc:.3f} +/- {sd:.3f}  (best published {leader_score})")

benchmarks = pd.DataFrame(benchmark_rows).set_index("Enzyme")
display(benchmarks)
benchmarks.to_csv("outputs/table3_tdc_benchmark.csv")

In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

fingerprint_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
herbs = pd.read_csv("data/herb_constituents.csv")
herb_fps = [fingerprint_gen.GetFingerprint(cp.standardise(s)) for s in herbs["SMILES"]]
X_herbs = cp.featurise(herbs["SMILES"].tolist())

rng = np.random.default_rng(42)
results = herbs[["Herb", "Compound"]].copy()
raw_probability = {}

for enzyme, (model, training_smiles) in herb_models.items():
    training_fps = [fingerprint_gen.GetFingerprint(Chem.MolFromSmiles(s))
                    for s in training_smiles]

    # domain threshold: 5th percentile of training molecules' own nearest-neighbour similarity
    sample = rng.choice(len(training_fps), 1000, replace=False)
    nearest = []
    for i in sample:
        sims = np.array(DataStructs.BulkTanimotoSimilarity(training_fps[i], training_fps))
        sims[i] = -1
        nearest.append(sims.max())
    threshold = np.percentile(nearest, 5)

    similarity = [np.array(DataStructs.BulkTanimotoSimilarity(fp, training_fps)).max()
                  for fp in herb_fps]
    probability = model.predict_proba(X_herbs)[:, 1]
    raw_probability[enzyme] = probability

    results[enzyme] = [
        round(p, 3) if s >= threshold else np.nan
        for p, s in zip(probability, similarity)
    ]
    results[f"{enzyme}_in_domain"] = np.array(similarity) >= threshold
    print(f"{enzyme}: threshold {threshold:.3f}, in domain {sum(np.array(similarity) >= threshold)}/{len(herbs)}")

raw_table = herbs[["Herb", "Compound"]].copy()
for enzyme_name, values in raw_probability.items():
    raw_table[enzyme_name] = values
raw_table.to_csv("outputs/tableS16_herb_raw_probabilities.csv", index=False)
results.to_csv("outputs/table4_herb_constituent_predictions.csv", index=False)
print("\nSaved to outputs/table4_herb_constituent_predictions.csv")

In [ ]:
enzyme_columns = list(cp.ENZYMES.keys())

scored = results.copy()
scored["Enzymes flagged"] = (scored[enzyme_columns] >= 0.5).sum(axis=1)
scored["Enzymes assessable"] = scored[enzyme_columns].notna().sum(axis=1)
scored["Highest probability"] = scored[enzyme_columns].max(axis=1)

top = (scored[scored["Enzymes flagged"] > 0]
       .sort_values(["Enzymes flagged", "Highest probability"], ascending=False))

print(f"Constituents flagged for at least one enzyme: {len(top)} of {len(scored)}")
display(top[["Herb", "Compound", "Enzymes flagged", "Enzymes assessable"] + enzyme_columns]
        .head(25).reset_index(drop=True))

scored.to_csv("outputs/table4_herb_constituent_predictions.csv", index=False)

### 3.1 Algorithm comparison on CYP3A4 (logistic regression and XGBoost)

Random 80/20 split of the cleaned CYP3A4 set, as in the first model comparison; the CYP3A4 features are those built above.

In [ ]:
import numpy as np

_stored = np.load("data/processed/all_enzymes/CYP3A4_features.npz")
X = _stored["X"].astype(np.float32)
y = _stored["y"]
descriptor_names = list(cp.DESCRIPTOR_NAMES)
data = {"X_fp": X[:, :2048], "X_desc": X[:, 2048:], "y": y, "descriptor_names": descriptor_names}

print(f"X (features): {X.shape[0]} molecules x {X.shape[1]} numbers each")
print(f"y (answers):  {y.shape[0]}")
print(f"Inhibitors:   {y.mean()*100:.1f}%")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training on: {len(y_train)} compounds")
print(f"Testing on:  {len(y_test)} compounds (never seen during training)")
print(f"Inhibitors — train {y_train.mean()*100:.1f}%, test {y_test.mean()*100:.1f}%")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42),
)

logreg.fit(X_train, y_train)

print("Logistic regression trained.")

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, confusion_matrix,
)


def evaluate(name, model):
    """Score a trained model on the test set and print the five metrics."""
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, prediction).ravel()

    scores = {
        "AUROC": roc_auc_score(y_test, probability),
        "AUPRC": average_precision_score(y_test, probability),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Balanced accuracy": balanced_accuracy_score(y_test, prediction),
    }

    print(f"--- {name} ---")
    for metric, value in scores.items():
        print(f"{metric:<20} {value:.3f}")
    print(f"\nCorrectly found inhibitors:     {tp}")
    print(f"Missed inhibitors:              {fn}")
    print(f"Correctly cleared non-inhibitors: {tn}")
    print(f"False alarms:                   {fp}")

    return scores, probability


logreg_scores, logreg_probability = evaluate("Logistic regression", logreg)

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb.fit(X_train, y_train)

xgb_scores, xgb_probability = evaluate("XGBoost", xgb)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

comparison = pd.DataFrame(
    {"Logistic regression": logreg_scores, "XGBoost": xgb_scores}
).round(3)
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for name, probability in [("Logistic regression", logreg_probability),
                          ("XGBoost", xgb_probability)]:
    fpr, tpr, _ = roc_curve(y_test, probability)
    axes[0].plot(fpr, tpr, label=name)

    precision, recall, _ = precision_recall_curve(y_test, probability)
    axes[1].plot(recall, precision, label=name)

axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guessing")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve")
axes[0].legend()

axes[1].axhline(y_test.mean(), color="k", linestyle="--", linewidth=1, label=f"No skill ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall (sensitivity)")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curve")
axes[1].legend()

plt.tight_layout()
fig.savefig("outputs/figS1_cyp3a4_logreg_vs_xgboost.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
comparison.to_csv("outputs/tableS2_cyp3a4_logreg_vs_xgboost.csv")

### 3.2 Feature attributions for CYP3A4 (SHAP)

SHAP values of the CYP3A4 model from 3.1 on 3,000 compounds, the substructures behind the most important fingerprint bits,
and the explanation of three flagged herb compounds.

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator, Draw
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

_stored = np.load("data/processed/all_enzymes/CYP3A4_features.npz")
X = _stored["X"].astype(np.float32)
y = _stored["y"]
data = {"X_fp": X[:, :2048], "X_desc": X[:, 2048:], "y": y}

descriptor_names = list(cp.DESCRIPTOR_NAMES)
feature_names = [f"bit_{i}" for i in range(2048)] + descriptor_names

model = xgb                                  # the CYP3A4 model trained in 3.1
structures = pd.read_csv("data/processed/all_enzymes/CYP3A4_standardised.csv")

print(f"Molecules: {X.shape[0]}   Features: {X.shape[1]}")

In [ ]:
sample = X[:3000]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

importance = np.abs(shap_values).mean(axis=0)
ranked = pd.DataFrame({
    "Feature": feature_names,
    "Mean impact": importance,
}).sort_values("Mean impact", ascending=False).reset_index(drop=True)

print("Top 15 features:")
display(ranked.head(15))

descriptors_only = ranked[ranked["Feature"].isin(descriptor_names)]
print(f"\nDescriptors contribute {descriptors_only['Mean impact'].sum() / importance.sum() * 100:.0f}%"
      f" of total impact, from just 10 of 2058 features.")

In [ ]:
shap.summary_plot(
    shap_values, sample, feature_names=feature_names,
    max_display=15, show=False,
)
plt.tight_layout()
plt.savefig("outputs/figS2_shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
logp_index = feature_names.index("MolLogP")

plt.figure(figsize=(7, 5))
plt.scatter(sample[:, logp_index], shap_values[:, logp_index], s=6, alpha=0.4)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("LogP")
plt.ylabel("Effect on prediction")
plt.title("How lipophilicity drives the model")
plt.tight_layout()
plt.savefig("outputs/figS3_shap_logp.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

top_bits = [int(f.split("_")[1]) for f in ranked["Feature"] if f.startswith("bit_")][:6]

rows, drawings = [], []
for bit in top_bits:
    on = data["X_fp"][:, bit] == 1
    rows.append({
        "Bit": bit,
        "Molecules with it": int(on.sum()),
        "Inhibitor rate with": round(float(y[on].mean()), 3),
        "Inhibitor rate without": round(float(y[~on].mean()), 3),
    })

    # find an example molecule containing this bit, and draw the substructure
    for smiles in structures["SMILES_std"]:
        mol = Chem.MolFromSmiles(smiles)
        output = rdFingerprintGenerator.AdditionalOutput()
        output.AllocateBitInfoMap()
        morgan_gen.GetFingerprint(mol, additionalOutput=output)
        bit_info = output.GetBitInfoMap()
        if bit in bit_info:
            drawings.append((bit, mol, bit_info))
            break

display(pd.DataFrame(rows))

Draw.DrawMorganBits(
    [(mol, bit, info) for bit, mol, info in drawings],
    molsPerRow=3, subImgSize=(250, 200),
    legends=[f"bit_{bit}" for bit, _, _ in drawings],
)

In [ ]:
largest_fragment = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()
descriptor_fns = {n: f for n, f in Descriptors.descList if n in descriptor_names}
descriptor_fns = {n: descriptor_fns[n] for n in descriptor_names}


def featurise_one(smiles):
    mol = uncharger.uncharge(largest_fragment.choose(Chem.MolFromSmiles(smiles)))
    fp = morgan_gen.GetFingerprintAsNumPy(mol)
    desc = [f(mol) for f in descriptor_fns.values()]
    return np.hstack([fp, desc]).astype(np.float32)


herbs = pd.read_csv("data/herb_constituents.csv")
to_explain = ["10-gingerol", "apigenin", "curcumin"]

for compound in to_explain:
    smiles = herbs.loc[herbs["Compound"] == compound, "SMILES"].iloc[0]
    x = featurise_one(smiles).reshape(1, -1)
    probability = model.predict_proba(x)[0, 1]
    values = explainer.shap_values(x)

    print(f"\n{compound}  —  predicted probability {probability:.3f}")
    shap.waterfall_plot(
        shap.Explanation(
            values=values[0], base_values=explainer.expected_value,
            data=x[0], feature_names=feature_names,
        ),
        max_display=10, show=False,
    )
    plt.tight_layout()
    plt.savefig(f"outputs/figS4_shap_{compound.replace('-', '_')}.png",
                dpi=300, bbox_inches="tight")
    plt.show()

## 4. Evaluation

Calibration (reliability diagrams and expected calibration error) and y-scrambling on the 80/20 scaffold split, and the
sensitivity of each enzyme model on that split at the 0.5 threshold. A y-scrambled model that reaches the performance of a real
model would indicate that the split, not the chemistry, drives the result; the assertion below checks this.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


import calibration as cal
import cyp_pipeline as cyp

ENZYMES = list(cyp.ENZYMES)
print("enzymes:", ", ".join(ENZYMES))

In [ ]:
import joblib

splits = {}
for enzyme in ENZYMES:
    features = np.load(PROJECT / "data" / "processed" / "all_enzymes" /
                       f"{enzyme}_features.npz", allow_pickle=True)
    frame = pd.read_csv(PROJECT / "data" / "processed" / "all_enzymes" /
                        f"{enzyme}_standardised.csv")
    scaffolds = cyp.scaffolds_of(frame["SMILES_std"].tolist())
    train_index, test_index = cyp.scaffold_split(scaffolds)
    splits[enzyme] = {"X": features["X"], "y": features["y"],
                      "train": train_index, "test": test_index}
    print(f"{enzyme:8s} {len(frame):6d} molecules   train {len(train_index):6d}   "
          f"test {len(test_index):5d}   inhibitors {100 * features['y'].mean():.1f}%")

In [ ]:
for enzyme, split in splits.items():
    model = cyp.train_model(split["X"][split["train"]], split["y"][split["train"]])
    split["probability"] = model.predict_proba(split["X"][split["test"]])[:, 1]
    split["y_test"] = split["y"][split["test"]]
    print(f"{enzyme}: predicted {len(split['probability'])} held-out molecules")

In [ ]:
summary = pd.DataFrame([
    {"Enzyme": enzyme, **cal.calibration_summary(split["y_test"], split["probability"])}
    for enzyme, split in splits.items()
]).round(3)

summary.to_csv(OUT / "tableS3_calibration.csv", index=False)
print("wrote outputs/tableS3_calibration.csv\n")
summary

In [ ]:
split_rows = []
for enzyme, split in splits.items():
    s = cyp.score(split["y_test"], split["probability"])   # threshold 0.5, as everywhere in the project
    split_rows.append({"Enzyme": enzyme, "Test compounds": int(len(split["y_test"])), "Sensitivity": round(s["Sensitivity"], 3),
                       "Specificity": round(s["Specificity"], 3), "AUROC": round(s["AUROC"], 3)})
split_table = pd.DataFrame(split_rows)
split_table.to_csv(OUT / "tableS5_scaffold_split_sensitivity.csv", index=False)
print("wrote outputs/tableS5_scaffold_split_sensitivity.csv\n")
split_table

In [ ]:
fig, axes = plt.subplots(1, len(ENZYMES), figsize=(4 * len(ENZYMES), 4), sharey=True)
for ax, enzyme in zip(axes, ENZYMES):
    split = splits[enzyme]
    bins = cal.reliability_curve(split["y_test"], split["probability"])
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
    ax.plot([b["mean_predicted"] for b in bins],
            [b["observed_frequency"] for b in bins],
            marker="o", label="model")
    ece = cal.expected_calibration_error(split["y_test"], split["probability"])
    ax.set_title(f"{enzyme}\nECE {ece:.3f}")
    ax.set_xlabel("mean predicted probability")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("observed inhibitor frequency")
axes[0].legend()
fig.suptitle("Reliability diagrams — below the diagonal means over-calling")
fig.tight_layout()

OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig1_calibration_reliability.png", dpi=150)
plt.show()

In [ ]:
constituents = pd.read_csv(OUT / "table4_herb_constituent_predictions.csv")

rows = []
for enzyme in ENZYMES:
    values = pd.to_numeric(constituents[enzyme], errors="coerce").dropna()
    bins = cal.reliability_curve(splits[enzyme]["y_test"], splits[enzyme]["probability"])
    flagged = values[values >= 0.70]
    # calibration gap in the bins the flagged constituents fall into
    gaps = [abs(b["observed_frequency"] - b["mean_predicted"])
            for b in bins if b["bin_low"] >= 0.70]
    rows.append({"Enzyme": enzyme,
                 "Constituents scored": len(values),
                 "Flagged (>= 0.70)": len(flagged),
                 "Median constituent p": round(values.median(), 3),
                 "Calibration gap above 0.70": round(np.mean(gaps), 3) if gaps else None})

pd.DataFrame(rows)

In [ ]:
scramble_rows = []
for enzyme, split in splits.items():
    real, scrambled = cal.y_scramble(
        split["X"], split["y"], split["train"], split["test"], n_repeats=10)
    split["scrambled"] = scrambled
    scramble_rows.append({
        "Enzyme": enzyme,
        "Real AUROC": round(real, 3),
        "Scrambled mean": round(float(np.mean(scrambled)), 3),
        "Scrambled max": round(float(np.max(scrambled)), 3),
        "Gap": round(real - float(np.mean(scrambled)), 3),
    })
    print(f"{enzyme:8s} real {real:.3f}   scrambled {np.mean(scrambled):.3f} "
          f"(max {np.max(scrambled):.3f})")

scramble = pd.DataFrame(scramble_rows)
scramble.to_csv(OUT / "tableS4_y_scrambling.csv", index=False)
print("\nwrote outputs/tableS4_y_scrambling.csv")
scramble

In [ ]:
worst = max(float(np.max(splits[e]["scrambled"])) for e in ENZYMES)
best_real = min(scramble["Real AUROC"])

assert worst < best_real, (
    f"a scrambled model reached AUROC {worst:.3f}, at or above the worst real model "
    f"({best_real:.3f}) — the models may be exploiting the split rather than chemistry")

print(f"Best scrambled run anywhere: {worst:.3f}")
print(f"Worst real model:            {best_real:.3f}")
print("\nPASS — no scrambled model comes close to any real model.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = range(len(ENZYMES))
ax.boxplot([splits[e]["scrambled"] for e in ENZYMES], positions=list(positions),
           widths=0.5, tick_labels=ENZYMES)
ax.scatter(positions, scramble["Real AUROC"], color="crimson", zorder=3,
           s=70, label="real model")
ax.axhline(0.5, color="grey", linestyle="--", alpha=0.7, label="chance")
ax.set_ylabel("AUROC on the real held-out labels")
ax.set_title("Y-scrambling: 10 shuffled-label models per enzyme vs the real model")
ax.legend()
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(OUT / "fig2_y_scrambling.png", dpi=150)
plt.show()

## 5. Herb-drug pairs and checks

### 5.1 Candidate pairs

A herb-drug pair is formed for every drug whose enzyme has at least one herb compound at or above 0.5 (the herb score is the
highest probability among its assessable compounds). Drug classes and narrow therapeutic index flags come from
`data/cyp_drug_substrates.csv` (FDA table of CYP substrates, Donnelly et al. 2025). Pairs are ordered into tiers by the drug side
and the strength of the prediction.

In [ ]:
import numpy as np
import pandas as pd

predictions = pd.read_csv("outputs/table4_herb_constituent_predictions.csv")
drugs = pd.read_csv("data/cyp_drug_substrates.csv")

ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]

print(f"Constituents: {len(predictions)}   Drug-enzyme pairs: {len(drugs)}")
print(f"NTI drugs: {sorted(drugs.loc[drugs['NTI'], 'Drug'].unique())}")
display(drugs.groupby(["Enzyme", "FDA_substrate_class"]).size().unstack(fill_value=0))

In [ ]:
signals = []
for herb, group in predictions.groupby("Herb"):
    for enzyme in ENZYMES:
        values = group[enzyme]
        assessable = values.notna().sum()
        if assessable == 0:
            signals.append({"Herb": herb, "Enzyme": enzyme, "Max probability": np.nan,
                            "Flagged constituents": 0, "Assessable": 0, "Driver": "out of domain"})
            continue
        best = values.idxmax()
        signals.append({
            "Herb": herb,
            "Enzyme": enzyme,
            "Max probability": round(values.max(), 3),
            "Flagged constituents": int((values >= 0.5).sum()),
            "Assessable": int(assessable),
            "Driver": group.loc[best, "Compound"],
        })

herb_signal = pd.DataFrame(signals)

strong = herb_signal[herb_signal["Flagged constituents"] > 0].sort_values(
    ["Max probability"], ascending=False)
print(f"Herb-enzyme combinations with at least one flagged constituent: {len(strong)}")
display(strong.head(20).reset_index(drop=True))

In [ ]:
flagged = herb_signal[herb_signal["Flagged constituents"] > 0]

pairs = flagged.merge(drugs, on="Enzyme", how="inner")

print(f"Candidate herb-drug pairs: {len(pairs)}")
print(f"Involving {pairs['Herb'].nunique()} herbs and {pairs['Drug'].nunique()} drugs")

In [ ]:
def tier(row):
    strong_signal = row["Max probability"] >= 0.70
    sensitive = row["FDA_substrate_class"] == "sensitive"

    if row["NTI"]:
        return "1 - NTI drug"
    if sensitive and strong_signal:
        return "2 - sensitive substrate, strong prediction"
    if sensitive or row["Serious_dose_dependent_toxicity"]:
        return "3 - sensitive substrate or known dose-dependent toxicity"
    return "4 - moderate substrate"


pairs["Priority"] = pairs.apply(tier, axis=1)
pairs = pairs.sort_values(
    ["Priority", "Max probability"], ascending=[True, False]).reset_index(drop=True)

print(pairs["Priority"].value_counts().sort_index().to_string())

pairs.to_csv("outputs/table5_herb_drug_pairs.csv", index=False)
print("\nSaved to outputs/table5_herb_drug_pairs.csv")

In [ ]:
columns = ["Herb", "Drug", "Enzyme", "Max probability", "Driver",
           "FDA_substrate_class", "Flagged constituents"]

print("=== TIER 1 — narrow therapeutic index drugs ===")
display(pairs[pairs["Priority"].str.startswith("1")][columns].reset_index(drop=True))

print("=== TIER 2 — sensitive substrates with a strong prediction ===")
display(pairs[pairs["Priority"].str.startswith("2")][columns].head(20).reset_index(drop=True))

In [ ]:
tier1 = pairs[pairs["Priority"].str.startswith("1")]
summary = (tier1.groupby(["Herb", "Drug"])["Max probability"].max()
           .reset_index().sort_values("Max probability", ascending=False))

print("Highest-priority herb-drug pairs for clinical investigation:\n")
for _, row in summary.iterrows():
    print(f"  {row['Herb']:<40} + {row['Drug']:<14} (p = {row['Max probability']:.3f})")

summary.to_csv("outputs/table6_priority_pairs.csv", index=False)

### 5.2 Comparison with published laboratory measurements (ChEMBL)

`data/chembl_measured.csv` is the frozen set of 248 ChEMBL activity records for the herb compounds on the five enzymes, retrieved
through the ChEMBL web service on 27 October 2026 (release not recorded). Records from the PubChem screening panel that the models were
trained on are excluded; only independent IC50 and Ki values (in nM) are compared. A measured value of 10 uM or lower counts as an inhibitor.
The optional re-download is described in the README (`scripts/fetch_chembl_activities.py`, `scripts/fetch_chembl_assays.py`).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

measured = pd.read_csv("data/chembl_measured.csv")

print(f"All measurements: {len(measured)}")
display(measured.groupby("source").size().to_frame("measurements"))

independent = measured[measured["independent"]]
print(f"\nIndependent: {len(independent)} measurements on "
      f"{independent['Compound'].nunique()} compounds")

In [ ]:
usable = independent[
    independent["standard_type"].isin(["IC50", "Ki"])
    & independent["units"].eq("nM")
    & independent["value"].notna()
]

potency = (usable.groupby(["Compound", "Enzyme", "standard_type"])
           .agg(median_nM=("value", "median"), measurements=("value", "size"))
           .reset_index()
           .rename(columns={"standard_type": "assay_type"}))
potency["median_uM"] = (potency["median_nM"] / 1000).round(1)

print(f"Compound-enzyme pairs with measured potency: {len(potency)}"
      f"  ({potency['Compound'].nunique()} compounds)")
display(potency.groupby("assay_type").size().to_frame("pairs"))
display(potency.sort_values("median_nM").head(12).reset_index(drop=True))

In [ ]:
ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]

predictions = pd.read_csv("outputs/table4_herb_constituent_predictions.csv")
long = (predictions.melt(id_vars=["Herb", "Compound"], value_vars=ENZYMES,
                         var_name="Enzyme", value_name="probability")
        .dropna(subset=["probability"])
        .drop_duplicates(["Compound", "Enzyme"]))

comparison = potency.merge(long[["Compound", "Enzyme", "probability"]],
                           on=["Compound", "Enzyme"], how="inner")
comparison = comparison.merge(
    predictions[["Compound", "Herb"]].drop_duplicates("Compound"), on="Compound", how="left")

comparison["Measured inhibitor"] = comparison["median_nM"] <= CHEMBL_CUTOFF_NM   # same criterion as the training labels
comparison["Predicted inhibitor"] = comparison["probability"] >= FLAG_THRESHOLD
comparison["Agree"] = comparison["Measured inhibitor"] == comparison["Predicted inhibitor"]


def report(frame, label):
    if len(frame) == 0 or frame["Measured inhibitor"].nunique() < 2:
        print(f"{label:<26} {len(frame):3d} pairs - too few to score")
        return
    print(f"{label:<26} {len(frame):3d} pairs  {frame['Compound'].nunique():2d} compounds  "
          f"agreement {frame['Agree'].mean()*100:3.0f}%  "
          f"AUROC {roc_auc_score(frame['Measured inhibitor'], frame['probability']):.3f}")


report(comparison, "IC50 + Ki together")
report(comparison[comparison["assay_type"] == "IC50"], "IC50 only (preferred)")
report(comparison[comparison["assay_type"] == "Ki"], "Ki only")

display(comparison.sort_values(["assay_type", "median_nM"])[
    ["Herb", "Compound", "Enzyme", "assay_type", "median_uM", "Measured inhibitor",
     "probability", "Predicted inhibitor", "Agree"]].reset_index(drop=True))

In [ ]:
ic50_only = comparison[comparison["assay_type"] == "IC50"]

rows = []
for cut_nM in [1_000, 5_000, 10_000, 20_000, 50_000]:
    label = ic50_only["median_nM"] <= cut_nM
    if label.nunique() < 2:
        continue
    rows.append({
        "Cut-off (uM)": cut_nM / 1000,
        "Measured inhibitors": int(label.sum()),
        "Agreement %": round(((ic50_only["probability"] >= 0.5) == label).mean() * 100),
        "AUROC": round(roc_auc_score(label, ic50_only["probability"]), 3),
    })

print("IC50 measurements only:")
display(pd.DataFrame(rows))

In [ ]:
# Which section 5.1 conclusions survive, and which weaken? (IC50 measurements only)
ic50_only = comparison[comparison["assay_type"] == "IC50"]

print("=== Confirmed potent by independent labs (<= 5 uM) ===")
display(ic50_only[ic50_only["median_nM"] <= 5000][
    ["Herb", "Compound", "Enzyme", "median_uM", "probability"]]
    .sort_values("median_uM").reset_index(drop=True))

print("=== Flagged by us, but measured WEAK (> 10 uM) - over-called ===")
display(ic50_only[(ic50_only["probability"] >= 0.5) & (ic50_only["median_nM"] > 10_000)][
    ["Herb", "Compound", "Enzyme", "median_uM", "probability"]]
    .sort_values("median_uM").reset_index(drop=True))

comparison.to_csv("outputs/table7_chembl_comparison.csv", index=False)
print("Saved to outputs/table7_chembl_comparison.csv")

In [ ]:
pairs = pd.read_csv("outputs/table5_herb_drug_pairs.csv")
measured = pd.read_csv("outputs/table7_chembl_comparison.csv")

# Drop any evidence columns from a previous run so this cell can be re-run safely.
pairs = pairs.drop(columns=["Evidence status", "Measured uM", "Measured assay"],
                   errors="ignore")

# One measurement per constituent + enzyme. Where both IC50 and Ki exist, prefer IC50.
measured["_rank"] = measured["assay_type"].map({"IC50": 0, "Ki": 1}).fillna(2)
measured = measured.sort_values("_rank").drop_duplicates(["Compound", "Enzyme"], keep="first")
lookup = measured.set_index(["Compound", "Enzyme"])
print("independently measured constituent-enzyme pairs:", len(lookup))


def evidence(row):
    key = (row["Driver"], row["Enzyme"])
    if key not in lookup.index:
        return "Untested"
    m = lookup.loc[key]
    if m["Measured inhibitor"] and m["Predicted inhibitor"]:
        return "Supported"
    if (not m["Measured inhibitor"]) and m["Predicted inhibitor"]:
        return "Contradicted"
    return "Measured, model did not flag"


pairs["Evidence status"] = pairs.apply(evidence, axis=1)
pairs["Measured uM"] = [lookup.loc[(d, e), "median_uM"] if (d, e) in lookup.index else pd.NA
                        for d, e in zip(pairs["Driver"], pairs["Enzyme"])]
pairs["Measured assay"] = [lookup.loc[(d, e), "assay_type"] if (d, e) in lookup.index else pd.NA
                           for d, e in zip(pairs["Driver"], pairs["Enzyme"])]
pairs.to_csv("outputs/table5_herb_drug_pairs.csv", index=False)

ORDER = {"Supported": 0, "Untested": 1, "Measured, model did not flag": 2, "Contradicted": 3}
priority = pairs[pairs["Priority"].str.startswith("1")].copy()
priority = priority[["Herb", "Drug", "Enzyme", "Driver", "Max probability", "Evidence status",
                     "Measured uM", "Measured assay", "FDA_substrate_class",
                     "Flagged constituents"]]
priority["_order"] = priority["Evidence status"].map(ORDER)
priority = (priority.sort_values(["_order", "Max probability"], ascending=[True, False])
            .drop(columns="_order").reset_index(drop=True))
priority.to_csv("outputs/table6_priority_pairs.csv", index=False)

counts_all = pairs["Evidence status"].value_counts()
counts_t1 = priority["Evidence status"].value_counts()
print(f"\nall {len(pairs)} pairs:\n{counts_all.to_string()}")
print(f"\n{len(priority)} Tier 1 (priority) pairs:\n{counts_t1.to_string()}")

# Expected counts of the reference run; a change means an upstream step moved.
assert len(priority) == 29, f"expected 29 priority pairs, got {len(priority)}"
assert (counts_all.get("Supported", 0), counts_all.get("Untested", 0),
        counts_all.get("Contradicted", 0)) == (33, 308, 82), counts_all.to_dict()
assert (counts_t1.get("Supported", 0), counts_t1.get("Untested", 0),
        counts_t1.get("Contradicted", 0)) == (3, 20, 6), counts_t1.to_dict()
print("\nPASS - reproduces the expected counts.")

### 5.3 Comparison with the Saudi Food and Drug Authority signal report

41 assessed herb-drug signals (`data/sfda_signals.csv`, Alghamdi et al. 2023). Only signals on herbs in the compound list, with a CYP-mediated
drug and at least one compound inside the applicability domain, can be judged by the model.

In [ ]:
import numpy as np
import pandas as pd

sfda = pd.read_csv("data/sfda_signals.csv")

print(f"Signals: {len(sfda)}   Herbs: {sfda['SFDA_herb'].nunique()}")
display(sfda["DIPS_rating"].value_counts().to_frame("signals"))

print("The 7 'probable' signals - the regulator's strongest conclusions:")
display(sfda[sfda["DIPS_rating"] == "Probable"][
    ["SFDA_herb", "Drug", "Mechanism_stated_in_paper"]].reset_index(drop=True))

In [ ]:
herbs = pd.read_csv("data/herb_constituents.csv")
our_herbs = sorted(herbs["Herb"].unique())

# map SFDA names onto ours
HERB_MAP = {
    "Turmeric": "Turmeric (Curcuma longa)",
    "Ginkgo biloba": "Ginkgo (Ginkgo biloba)",
    "Green tea": "Green tea (Camellia sinensis)",
}

sfda["Our_herb"] = sfda["SFDA_herb"].map(HERB_MAP)
covered = sfda[sfda["Our_herb"].notna()]

print("SFDA herbs we cover:", sorted(HERB_MAP))
print("SFDA herbs we do NOT cover:",
      sorted(set(sfda["SFDA_herb"]) - set(HERB_MAP)))
print(f"\nSignals on herbs we cover: {len(covered)} of {len(sfda)}")
display(covered.groupby(["SFDA_herb", "DIPS_rating"]).size().unstack(fill_value=0))

In [ ]:
drugs = pd.read_csv("data/cyp_drug_substrates.csv")

# FDA lists S-warfarin; SFDA says warfarin
drug_lookup = drugs.copy()
drug_lookup["match_name"] = drug_lookup["Drug"].str.replace("^S-", "", regex=True).str.lower()

matched = covered.merge(
    drug_lookup[["match_name", "Enzyme", "FDA_substrate_class", "NTI"]],
    left_on=covered["Drug"].str.lower(), right_on="match_name", how="left"
).drop(columns="match_name")

in_scope = matched[matched["Enzyme"].notna()]
out_of_scope = matched[matched["Enzyme"].isna()]

print(f"CYP-mediated (in scope):     {len(in_scope)}")
print(f"Not on the FDA CYP list:     {len(out_of_scope)}")
print()
display(in_scope[["SFDA_herb", "Drug", "DIPS_rating", "Enzyme",
                  "FDA_substrate_class", "NTI"]].reset_index(drop=True))

print("Out of scope - our model can never detect these:")
display(out_of_scope[["SFDA_herb", "Drug", "DIPS_rating",
                      "Mechanism_stated_in_paper"]].reset_index(drop=True))

In [ ]:
predictions = pd.read_csv("outputs/table4_herb_constituent_predictions.csv")
ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]

signal_rows = []
for _, row in in_scope.iterrows():
    enzyme = row["Enzyme"]
    group = predictions[predictions["Herb"] == row["Our_herb"]]
    values = group[enzyme]
    assessable = int(values.notna().sum())

    signal_rows.append({
        "SFDA herb": row["SFDA_herb"],
        "Drug": row["Drug"],
        "DIPS": row["DIPS_rating"],
        "Enzyme": enzyme,
        "Constituents in domain": assessable,
        "Our max probability": round(values.max(), 3) if assessable else np.nan,
        "Driver": group.loc[values.idxmax(), "Compound"] if assessable else "out of domain",
        "Model flagged": bool(assessable and values.max() >= 0.5),
        "Low confidence (CYP2D6)": enzyme == "CYP2D6",
    })

results = pd.DataFrame(signal_rows)
display(results)

In [ ]:
assessable = results[results["Constituents in domain"] > 0]
flagged = assessable[assessable["Model flagged"]]
missed = assessable[~assessable["Model flagged"]]

print("=== Denominators ===")
print(f"All SFDA assessed signals                 : {len(sfda)}")
print(f"  on herbs we cover                       : {len(covered)}")
print(f"  AND drug is CYP-mediated (FDA list)     : {len(in_scope)}")
print(f"  AND in applicability domain             : {len(assessable)}   <- what we can judge")
print()
print(f"Of those {len(assessable)}: flagged by model {len(flagged)}, missed {len(missed)}")
print()
print("=== Flagged ===")
display(flagged[["SFDA herb", "Drug", "DIPS", "Enzyme", "Our max probability", "Driver"]]
        .reset_index(drop=True))
print("=== Missed ===")
display(missed[["SFDA herb", "Drug", "DIPS", "Enzyme", "Our max probability", "Driver"]]
        .reset_index(drop=True))

results.to_csv("outputs/table8_sfda_comparison.csv", index=False)
print("Saved to outputs/table8_sfda_comparison.csv")

In [ ]:
mechanism = in_scope[["SFDA_herb", "Drug", "DIPS_rating", "Mechanism_stated_in_paper"]].copy()
mechanism["Mechanism_stated_in_paper"] = mechanism["Mechanism_stated_in_paper"].fillna("not stated")
mechanism["Paper says CYP"] = mechanism["Mechanism_stated_in_paper"].str.contains("CYP")
mechanism = mechanism.merge(
    results[["SFDA herb", "Drug", "Our max probability", "Model flagged"]],
    left_on=["SFDA_herb", "Drug"], right_on=["SFDA herb", "Drug"], how="left").drop(columns="SFDA herb")

display(mechanism)

cyp_signals = mechanism[mechanism["Paper says CYP"]]
print(f"Signals the paper attributes to CYP inhibition: {len(cyp_signals)}")
for _, r in cyp_signals.iterrows():
    verdict = "FLAGGED" if r["Model flagged"] else "MISSED"
    print(f"  {r['SFDA_herb']} + {r['Drug']} ({r['DIPS_rating']}): "
          f"our p = {r['Our max probability']:.3f} -> {verdict}")

### 5.4 Prior documentation in the herb guide and the review sheet

For each priority pair, whether the Memorial Sloan Kettering "About Herbs" guide names the enzyme or the drug
(`data/c3_mskcc_lookup.csv`, checked 29 October 2026), the tier rule, the sensitivity of the result to the matching rule, the
agreement with a second independent lookup of the same guide, and the review workbook.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


pairs = pd.read_csv(OUT / "table6_priority_pairs.csv")
substrates = pd.read_csv(DATA / "cyp_drug_substrates.csv")
# keep_default_na=False: blank cells stay "" and the text "None"/"NA" is never turned into NaN
lookup = pd.read_csv(DATA / "c3_mskcc_lookup.csv", keep_default_na=False)

# ASSERTION 1 - the pair table is exactly what the whole analysis was built on
assert len(pairs) == 29, f"table6_priority_pairs.csv has {len(pairs)} rows, expected 29"
assert pairs["Herb"].nunique() == 12, f"{pairs['Herb'].nunique()} distinct herbs, expected 12"

# ASSERTION 2 - the C3 Prior documentation lookup covers exactly those herbs, once each
assert lookup["Herb"].is_unique, "a herb appears more than once in c3_mskcc_lookup.csv"
assert set(lookup["Herb"]) == set(pairs["Herb"]), (
    "herb sets differ - only in lookup: "
    f"{sorted(set(lookup['Herb']) - set(pairs['Herb']))}; only in pairs: "
    f"{sorted(set(pairs['Herb']) - set(lookup['Herb']))}")

print(f"OK  29 pairs, 12 herbs; lookup covers the same 12 herbs")
print("Evidence status columns present:", "Evidence status" in pairs.columns,
      "")

In [ ]:
SUBSTRATE_LABEL = {"sensitive": "Sensitive (≥5-fold)", "moderate": "Moderate (2 to 5-fold)"}

review = pairs[["Herb", "Drug", "Enzyme", "Driver", "Max probability", "Evidence status"]].merge(
    # "NTI" is the column's name in the upstream file: narrow therapeutic index
    substrates[["Drug", "Enzyme", "FDA_substrate_class", "NTI"]],
    on=["Drug", "Enzyme"], how="left", validate="m:1")        # m:1 -> no duplicated substrate rows

assert len(review) == 29, "the substrate join changed the number of rows"
assert review["FDA_substrate_class"].notna().all(), "a pair has no row in cyp_drug_substrates.csv"

review["Substrate_sensitivity"] = review["FDA_substrate_class"].map(SUBSTRATE_LABEL)
review["Narrow_therapeutic_index"] = review["NTI"].map({True: "Yes", False: "No"})
assert review["Substrate_sensitivity"].notna().all() and review["Narrow_therapeutic_index"].notna().all(), \
    "unmapped Substrate sensitivity or Narrow therapeutic index value"

# the substrate table agrees with the class already stored in table6_priority_pairs.csv
assert (review["FDA_substrate_class"].values == pairs["FDA_substrate_class"].values).all(), \
    "cyp_drug_substrates.csv disagrees with table6_priority_pairs.csv on substrate class"

assert (review["Narrow_therapeutic_index"] == "Yes").all(), \
    "Narrow therapeutic index is not Yes for all 29 pairs"
print("WARNING: Narrow therapeutic index is Yes for all 29 pairs, so it cannot separate them.")
print("         It is shown in the sheet but does NOT enter the tier rule.\n")
print(review.groupby(["Drug", "Enzyme"])[["Substrate_sensitivity", "Narrow_therapeutic_index"]].first().to_string())

In [ ]:
import sys
import prior_documentation as pdoc

LK = lookup.set_index("Herb")
MENTIONED, NOT_MENTIONED = pdoc.MENTIONED, pdoc.NOT_MENTIONED
parts, code_of, norm_drug = pdoc.parts, pdoc.code_of, pdoc.norm_drug

def c3_basis(herb, drug, enzyme, rule=1):
    """Why a pair is (or is not) mentioned (Prior documentation). rule 1 = default, 2 = strict, 3 = loose."""
    return pdoc.basis(LK.loc[herb], drug, enzyme, rule)

TIER_NAMES = {1: "Tier 1 - novel priority", 2: "Tier 2 - secondary", 3: "Tier 3 - low"}

def tier_of(substrate_sensitivity, prior_documentation):
    """Tier 1 = Sensitive AND Not mentioned; Tier 2 = exactly one of those two; Tier 3 = neither. No points."""
    n_true = (substrate_sensitivity == SUBSTRATE_LABEL["sensitive"]) + (prior_documentation == NOT_MENTIONED)
    return TIER_NAMES[{2: 1, 1: 2, 0: 3}[int(n_true)]]

def score(df, rule):
    out = df.copy()
    out["C3_basis"] = [c3_basis(h, d, e, rule) for h, d, e in zip(df.Herb, df.Drug, df.Enzyme)]
    out["C3_status"] = np.where(out["C3_basis"].map(pdoc.is_mentioned), MENTIONED, NOT_MENTIONED)
    out["Tier"] = [tier_of(s, c) for s, c in zip(out["Substrate_sensitivity"], out["C3_status"])]
    return out

r1 = score(review, 1)
r1["Enzyme_named_in_monograph"] = [
    "Yes" if code_of(e) in {code_of(x) for x in parts(LK.loc[h, "Enzymes_named"])} else "No"
    for h, e in zip(r1.Herb, r1.Enzyme)]
r1["Drug_named_in_monograph"] = [
    "Yes" if norm_drug(d) in {norm_drug(x) for x in parts(LK.loc[h, "Drugs_named_among_study_drugs"])} else "No"
    for h, d in zip(r1.Herb, r1.Drug)]

print(r1["C3_basis"].value_counts().to_string())
print("\nPrior documentation:", r1["C3_status"].value_counts().to_dict())
print("Tiers   :", r1["Tier"].value_counts().sort_index().to_dict())

In [ ]:
token_in = pdoc.token_in
pdoc.self_test_token_in()          # the matcher's positive and negative self-tests (CYP2C19 must not match 2C9, ...)

# ASSERTION 3 - every pair has a Prior documentation value
assert r1["C3_status"].isin([MENTIONED, NOT_MENTIONED]).all() and len(r1) == 29, \
    "a pair has no Prior documentation value"
assert set(r1["C3_basis"]) <= {"enzyme named - Herb-Drug Interactions", "enzyme named - case report only",
                               "drug named", "enzyme not named", "no monograph"}, \
    "unexpected Prior documentation basis"

# ASSERTION 4 - every Mentioned pair is backed by the structured columns of the lookup: the enzyme it was matched on
#               is listed in Enzymes_named (case-report-only enzymes in Enzymes_case_report_only too), or, for drug-only
#               pairs, the drug is listed in Drugs_named_among_study_drugs
for _, row in r1[r1["C3_status"] == MENTIONED].iterrows():
    rec = LK.loc[row.Herb]
    assert rec["In_MSKCC"] == "Yes", f"{row.Herb} x {row.Drug}: Mentioned but the herb has no page"
    if row.C3_basis.startswith("enzyme named"):
        named = {code_of(e) for e in parts(rec["Enzymes_named"])}
        case_only = {code_of(e) for e in parts(rec["Enzymes_case_report_only"])}
        assert code_of(row.Enzyme) in named, f"{row.Herb} x {row.Enzyme}: enzyme not in Enzymes_named"
        assert (code_of(row.Enzyme) in case_only) == (row.C3_basis == "enzyme named - case report only"), \
            f"{row.Herb} x {row.Enzyme}: case-report flag does not match the basis"
    else:
        assert norm_drug(row.Drug) in {norm_drug(d) for d in parts(rec["Drugs_named_among_study_drugs"])}, \
            f"{row.Herb} x {row.Drug}: drug not in Drugs_named_among_study_drugs"

# ASSERTION 6 - the default rule reproduces the agreed counts exactly.
# tier (they were 5 / 12 / 12 under the retired point scores; the Tier 1 set is identical - assertion 7).
n_doc = int((r1["C3_status"] == MENTIONED).sum())
n_not = int((r1["C3_status"] == NOT_MENTIONED).sum())
tiers = r1["Tier"].value_counts().to_dict()
assert (n_doc, n_not) == (12, 17), f"Rule 1: Mentioned/Not mentioned = {n_doc}/{n_not}, expected 12/17"
assert tiers == {"Tier 1 - novel priority": 5, "Tier 2 - secondary": 19, "Tier 3 - low": 5}, \
    f"Rule 1 tier counts are {tiers}, expected 5 / 19 / 5"
# the tier is recomputed here a second way, straight from the rule as written, for every pair
for _, row in r1.iterrows():
    sensitive, not_mentioned = row.FDA_substrate_class == "sensitive", row.C3_status == NOT_MENTIONED
    expected = 1 if (sensitive and not_mentioned) else 3 if (not sensitive and not not_mentioned) else 2
    assert row.Tier == TIER_NAMES[expected], f"{row.Herb} x {row.Drug}: tier {row.Tier} breaks the rule"

# ASSERTION 7 - and the five Tier 1 pairs are exactly these (unchanged from the point-score version)
short = lambda herb: herb.split(" (")[0]
tier1 = {(short(h), d) for h, d in zip(r1[r1.Tier.str.startswith("Tier 1")].Herb,
                                         r1[r1.Tier.str.startswith("Tier 1")].Drug)}
expected_t1 = {("Parsley", "tacrolimus"), ("Parsley", "sirolimus"), ("Parsley", "everolimus"),
               ("Ginger", "sirolimus"), ("Ginger", "everolimus")}
assert tier1 == expected_t1, f"Tier 1 pairs are {sorted(tier1)}"

# ASSERTION 8 - the naming rule behaves as intended
def status(herb, drug):
    m = r1[(r1.Herb.str.startswith(herb)) & (r1.Drug == drug)]
    assert len(m) == 1, f"{herb} x {drug} matched {len(m)} rows"
    return m["C3_status"].iloc[0]
assert status("Green tea", "S-warfarin") == NOT_MENTIONED, "green tea x S-warfarin should be Not mentioned"
assert status("Ginger", "tacrolimus") == MENTIONED, "ginger x tacrolimus should be Mentioned"
assert status("Ginger", "sirolimus") == NOT_MENTIONED, "ginger x sirolimus should be Not mentioned"

# ASSERTION 10 (added with the Drugs_named_all column) - the column that is MATCHED must be exactly the
#               study drugs found in the all-drugs column, so the two columns cannot drift apart
STUDY_DRUGS = {norm_drug(d) for d in pairs["Drug"]}
for herb, rec in LK.iterrows():
    all_named = {norm_drug(d) for d in parts(rec["Drugs_named_all"])}
    scored_named = {norm_drug(d) for d in parts(rec["Drugs_named_among_study_drugs"])}
    assert scored_named == all_named & STUDY_DRUGS, (
        f"{herb}: study-drugs column {sorted(scored_named)} != study drugs in Drugs_named_all "
        f"{sorted(all_named & STUDY_DRUGS)}")
    if rec["In_MSKCC"] == "No":
        assert not all_named and not scored_named, f"{herb}: no monograph but drugs recorded"

print("PASS  assertions 3, 4, 6, 7, 8, 10")
print(f"      Rule 1: Mentioned {n_doc} / Not mentioned {n_not};  tiers {tiers}")
print("      Tier 1:", ", ".join(f"{h} + {d}" for h, d in sorted(tier1)))

In [ ]:
RULES = {1: "Rule 1 (default)", 2: "Rule 2 (strict: no case-report-only enzymes)",
         3: "Rule 3 (loose: a drug class names its members)"}
scored = {k: score(review, k) for k in RULES}

summary = pd.DataFrame([{
    "Rule": RULES[k],
    "Mentioned": int((s["C3_status"] == MENTIONED).sum()),
    "Not mentioned": int((s["C3_status"] == NOT_MENTIONED).sum()),
    "Tier 1": int(s["Tier"].str.startswith("Tier 1").sum()),
    "Tier 2": int(s["Tier"].str.startswith("Tier 2").sum()),
    "Tier 3": int(s["Tier"].str.startswith("Tier 3").sum()),
} for k, s in scored.items()])

key = ["Herb", "Drug", "Enzyme"]
wide = review[key].copy()
for k, s in scored.items():
    wide[f"C3_rule{k}"] = s["C3_status"].values
    wide[f"Tier_rule{k}"] = s["Tier"].values
for k in (2, 3):
    wide[f"Differs_rule{k}"] = wide[f"Tier_rule{k}"] != wide["Tier_rule1"]

differs = pd.concat([
    wide.loc[wide[f"Differs_rule{k}"], key + ["Tier_rule1", f"Tier_rule{k}"]]
        .rename(columns={f"Tier_rule{k}": "Tier under this rule"}).assign(Rule=RULES[k])
    for k in (2, 3)], ignore_index=True)[["Rule"] + key + ["Tier_rule1", "Tier under this rule"]]

# ASSERTION 9
t1 = {k: frozenset(zip(s[s.Tier.str.startswith("Tier 1")].Herb, s[s.Tier.str.startswith("Tier 1")].Drug))
      for k, s in scored.items()}
d2, d3 = summary.loc[1, ["Mentioned", "Not mentioned"]], summary.loc[2, ["Mentioned", "Not mentioned"]]
assert (d2["Mentioned"], d2["Not mentioned"]) == (10, 19), f"Rule 2 gives {tuple(d2)}, expected (10, 19)"
assert (d3["Mentioned"], d3["Not mentioned"]) == (14, 15), f"Rule 3 gives {tuple(d3)}, expected (14, 15)"
assert t1[1] == t1[2] == t1[3], "the Tier 1 set is not identical under all three rules"

wide.to_csv(OUT / "tableS12_prior_documentation_rule_sensitivity.csv", index=False)
print("PASS  assertion 9;  wrote outputs/tableS12_prior_documentation_rule_sensitivity.csv\n")
print(summary.to_string(index=False))
print("\nPairs whose tier differs from Rule 1:")
print(differs.to_string(index=False))
print("\nTier 1 set identical under all three rules:", t1[1] == t1[2] == t1[3])

In [ ]:
t1_rows = r1[r1.Tier.str.startswith("Tier 1")]
print("Tier 1 pairs with their independent evidence status (NOT part of the tier rule):\n")
print(t1_rows[["Herb", "Drug", "Enzyme", "Driver", "Max probability", "Evidence status", "C3_basis"]]
      .to_string(index=False))
print("\nEvidence status of the 5 Tier 1 pairs:", t1_rows["Evidence status"].value_counts().to_dict())
print("Evidence status of all 29 pairs      :", r1["Evidence status"].value_counts().to_dict())

In [ ]:
indep_path = DATA / "STEP_8c_c3_independent_pairs.csv"
assert indep_path.exists(), f"{indep_path.name} is missing"
indep = pd.read_csv(indep_path, keep_default_na=False)
indep["C3_independent"] = indep["C3_independent"].map({"Documented": MENTIONED, "Not documented": NOT_MENTIONED})
assert indep["C3_independent"].notna().all(), "unexpected value in the independent lookup"

merged = r1[key + ["C3_status", "C3_basis"]].merge(
    indep[key + ["C3_independent", "Basis"]], on=key, how="outer", indicator=True)
assert (merged["_merge"] == "both").all() and len(merged) == 29, (
    "the two tables do not cover the same 29 pairs:\n"
    + merged[merged["_merge"] != "both"].to_string())

agree = merged.rename(columns={"C3_status": "C3_notebook", "C3_basis": "C3_basis_notebook",
                               "Basis": "Basis_independent"}).drop(columns="_merge")
agree["Agree"] = agree["C3_notebook"] == agree["C3_independent"]
agree = agree[key + ["C3_notebook", "C3_independent", "Agree", "C3_basis_notebook", "Basis_independent"]]
agree.to_csv(OUT / "tableS11_prior_documentation_agreement.csv", index=False)

n_agree, n_dis = int(agree["Agree"].sum()), int((~agree["Agree"]).sum())
print(f"AGREE: {n_agree} of 29     DISAGREE: {n_dis} of 29     (wrote outputs/tableS11_prior_documentation_agreement.csv)")
if n_dis:
    pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)
    print("\nEVERY DISAGREEMENT, IN FULL - none has been resolved:\n")
    print(agree[~agree["Agree"]].to_string(index=False))
else:
    print("No disagreements.")

In [ ]:
MSKCC_BASE = "https://www.mskcc.org/cancer-care/integrative-medicine/herbs/"
GUIDE = "Memorial Sloan Kettering herb guide"        # the guide is always written out in the workbook
ENZYME_LABEL = "Cytochrome P450 (CYP) enzyme"
CRITERIA_NAMES = {"1": "Substrate sensitivity", "2": "Narrow therapeutic index", "3": "Prior documentation"}
SHORT_RULE = {1: "Rule 1 (default)", 2: "Rule 2 (strict)", 3: "Rule 3 (loose)"}

def spell_out_guide(value):
    return value.replace("MSKCC", GUIDE) if isinstance(value, str) else value

# ------------------------------------------------------------------ Review sheet ------------------
REVIEW_COLUMNS = {                       # internal name -> header shown in the workbook
    "Herb": "Herb",
    "Drug": "Drug",
    "Enzyme": ENZYME_LABEL,
    "Driver": "Herb constituent driving the prediction",
    "Max probability": "Maximum predicted probability",
    "Evidence status": "Independent evidence status (Supported, Untested or Contradicted)",
    "Substrate_sensitivity": "Substrate sensitivity",
    "Narrow_therapeutic_index": "Narrow therapeutic index",
    "C3_display": f"Prior documentation ({GUIDE})",
    "C3_basis": "Prior documentation basis",
    "Enzyme_named_in_monograph": "Enzyme named on the herb's page",
    "Drug_named_in_monograph": "Drug named on the herb's page",
    "Tier": "Tier (rule-based; see the Method sheet)",
}
r1["C3_display"] = [pdoc.display_label(b) for b in r1["C3_basis"]]       # the three displayed labels
review_sheet = (r1.sort_values(["Tier", "Max probability"], ascending=[True, False])
                  .reset_index(drop=True)[list(REVIEW_COLUMNS)].copy())
review_sheet = review_sheet.rename(columns=REVIEW_COLUMNS)

# ------------------------------------------------------------------ C3 evidence sheet ------------
EVIDENCE_COLUMNS = {
    "Herb": "Herb",
    "In_MSKCC": f"Has a page in the {GUIDE}",
    "Monograph_URL": "Page web address",
    "Enzymes_named": ENZYME_LABEL + "s named on the page",
    "Enzymes_case_report_only": "Enzymes named only in a case report",
    "Drugs_named_among_study_drugs": "Drugs named among the six study drugs (the only drug column used for the 29 pairs)",
    "Drugs_named_all": "All drugs named in the Herb-Drug Interactions section (reference; used for the pairs outside the 29)",
    "Classes_named": "Drug classes named (a class does not count as naming a drug)",
    "Basis_note": "Basis note",
    "Page_last_updated": "Page last updated",
    "Date_checked": "Date checked",
}
evidence_sheet = lookup.copy()
evidence_sheet["Monograph_URL"] = np.where(lookup["Monograph_slug"] != "", MSKCC_BASE + lookup["Monograph_slug"], "")
evidence_sheet["Classes_named"] = evidence_sheet["Classes_named"].str.replace(
    "NSAIDs", "Nonsteroidal anti-inflammatory drugs (NSAIDs)", regex=False)
evidence_sheet = evidence_sheet[list(EVIDENCE_COLUMNS)].map(spell_out_guide).rename(columns=EVIDENCE_COLUMNS)

# ------------------------------------------------------------------ C3 sensitivity sheet ----------
sens_summary = pd.DataFrame({"Rule": summary["Rule"]})
for label in pdoc.DISPLAY_LABELS:
    sens_summary[f"Pairs: {label}"] = [int(sum(pdoc.display_label(b) == label for b in s["C3_basis"])) for s in scored.values()]
sens_summary["Pairs in Tier 1 - novel priority"] = summary["Tier 1"]
sens_summary["Pairs in Tier 2 - secondary"] = summary["Tier 2"]
sens_summary["Pairs in Tier 3 - low"] = summary["Tier 3"]
sens_differs = differs.rename(columns={"Enzyme": ENZYME_LABEL, "Tier_rule1": "Tier under Rule 1 (default)"})
SENS_WIDE_COLUMNS = {"Herb": "Herb", "Drug": "Drug", "Enzyme": ENZYME_LABEL}
for k in (1, 2, 3):
    SENS_WIDE_COLUMNS[f"C3_rule{k}"] = f"Prior documentation under {SHORT_RULE[k]}"
    SENS_WIDE_COLUMNS[f"Tier_rule{k}"] = f"Tier under {SHORT_RULE[k]}"
for k in (2, 3):
    SENS_WIDE_COLUMNS[f"Differs_rule{k}"] = f"Tier differs from Rule 1 under {SHORT_RULE[k]}"
sens_wide = wide.copy()
for k in (1, 2, 3):
    sens_wide[f"C3_rule{k}"] = [pdoc.display_label(b) for b in scored[k]["C3_basis"]]
for k in (2, 3):
    sens_wide[f"Differs_rule{k}"] = sens_wide[f"Differs_rule{k}"].map({True: "Yes", False: "No"})
sens_wide = sens_wide[list(SENS_WIDE_COLUMNS)].rename(columns=SENS_WIDE_COLUMNS)
SENS_TITLE = (f"Prior documentation ({GUIDE}): number of pairs Mentioned and Not mentioned, and tier counts "
              "under the rule-based tier, under each rule (computed by the notebook)")

# ------------------------------------------------------------------ dropped criterion sheet -------
DROPPED_SHEET = "Dropped - regional availability"
dropped_sheet = pd.DataFrame([
    ("Saudi Arabia", "Free national medicines listing"), ("Jordan", "Free national medicines listing"),
    ("Qatar", "Free national medicines listing"), ("Lebanon", "Free national medicines listing"),
    ("Iraq", "Free national medicines listing"),
    ("United Arab Emirates", "Paid"), ("Iran", "Geo-restricted"),
    ("Kuwait", "None found"), ("Bahrain", "None found"), ("Oman", "None found"), ("Egypt", "None found"),
], columns=["Country", "National medicines listing"])
DROPPED_NOTE = ("Dropped criterion: regional availability. Documentation only - not part of the tier rule, and it carries no criterion number. "
                "It was dropped (see the Method sheet). This is the country-access survey kept as a separate finding, recorded "
                "as supplied by the project owner on 2026-10-29; it was not re-verified in this notebook.")

# ------------------------------------------------------------------ Method sheet ------------------
n_no_mono = int((r1["C3_basis"] == "no monograph").sum())
t1c = ", ".join(f"{n} {k}" for k, n in t1_rows["Evidence status"].value_counts().items())
sub_src = " / ".join(substrates["Substrate_source"].dropna().unique())
nti_src = " / ".join(substrates["NTI_source"].dropna().unique())
for old, new in [("FDA, Drug", "Food and Drug Administration (FDA), Drug")]:               # first use of FDA
    assert old in sub_src, f"substrate source text changed: {sub_src!r}"
    sub_src = sub_src.replace(old, new, 1)
for old, new in [("PMID", "PubMed identifier"), ("Clin Pharmacol Ther", "Clinical Pharmacology and Therapeutics"),
                 ("FDA publishes no consolidated public NTI list", "The FDA publishes no consolidated public list of narrow-therapeutic-index drugs")]:
    assert old in nti_src, f"narrow-therapeutic-index source text changed: {nti_src!r}"
    nti_src = nti_src.replace(old, new, 1)
rule_counts = "; ".join(f"{SHORT_RULE[k]} {int(s['Tier'].str.startswith('Tier 1').sum())} / {int(s['Tier'].str.startswith('Tier 2').sum())} / "
                        f"{int(s['Tier'].str.startswith('Tier 3').sum())}" for k, s in scored.items())

METHOD = [
 ("Title", "Pharmacist review sheet: how the labels and the tier were built"),
 ("Provenance", "Written by phytocyp_pipeline.ipynb, from scratch on every run, from outputs/table6_priority_pairs.csv, "
                "data/cyp_drug_substrates.csv and data/c3_mskcc_lookup.csv, using the matcher in src/prior_documentation.py. Anything typed into this "
                "workbook by hand is lost on re-run - keep reviewer decisions in a separate file. The Glossary sheet defines every "
                "abbreviation and label used here."),
 ("The three labels and the tier",
  "Each pair carries three labels: C1 Substrate sensitivity, C2 Narrow therapeutic index and C3 Prior documentation. There are no point scores and no "
  "total. The tier is a rule applied to the labels."),
 ("Tier rule",
  "Tier 1 - novel priority = Sensitive AND Not mentioned. Tier 2 - secondary = exactly one of those two is true (Sensitive and Mentioned, or Moderate "
  "and Not mentioned). Tier 3 - low = neither is true (Moderate and Mentioned). This tier exists only in this spreadsheet; the pair table "
  "(outputs/table9_pair_table.csv) carries the labels and no tier."),
 ("Why a rule with no weights",
  "No validated scheme for ranking unreported herb-drug pairs was found in the literature search for this project. The published schemes grade "
  "interactions that have already been reported, by evidence and seriousness. De Smet (2007), as applied by Pochet et al. (2022, a study of VigiBase, "
  "the World Health Organization's database of suspected adverse drug reaction reports), gives an evidence-quality code from 0 to 4 and a seriousness "
  "code from A to F, and reports the two side by side rather than adding them. BABINE (Cnudde et al., 2024) adds points inside each of its sections, "
  "converts each section total to a category (Poor, Average or Good) and combines severity and reliability in a matrix. Because no validated weights "
  "exist for unreported pairs, this sheet uses a transparent rule with no weights. An earlier version of this sheet used 2 / 1 / 2 point weights; "
  "those were the author's own design choice with no external source, and they have been removed."),
 ("C1 Substrate sensitivity",
  "How strongly the drug depends on the pair's cytochrome P450 (CYP) enzyme, labelled Sensitive (≥5-fold) or Moderate (2 to 5-fold). Using the "
  "definitions of the Food and Drug Administration (FDA): a sensitive substrate is a drug whose exposure (area under the concentration-time curve) "
  "rises 5-fold or more with a strong inhibitor of that enzyme; a moderate substrate is one whose exposure rises 2-fold to less than 5-fold. "
  "Source: the FDA substrate class column of data/cyp_drug_substrates.csv. " + sub_src),
 ("C2 Narrow therapeutic index",
  "Whether the drug has a narrow therapeutic index: a small gap between a helpful dose and a harmful one, so small changes in blood level matter. "
  "Labelled Yes or No. Source: the narrow-therapeutic-index column of data/cyp_drug_substrates.csv. " + nti_src),
 ("C2 Narrow therapeutic index does not enter the tier rule",
  f"It is Yes for all {int((r1['Narrow_therapeutic_index'] == 'Yes').sum())} pairs: every one of the six drugs has a narrow therapeutic index, so it "
  "cannot separate any pair from any other. It is shown, but it is not in the rule."),
 (f"C3 Prior documentation ({GUIDE})",
  f"Whether the herb's page in the {GUIDE} ('About Herbs') already names this enzyme or this drug. Labelled Mentioned or Not mentioned. Source: the "
  f"guide's herb pages ({MSKCC_BASE}), checked 2026-10-29, recorded in data/c3_mskcc_lookup.csv. The full A-Z listing was enumerated (308 pages) so "
  "that absence could be established."),
 ("Prior documentation labels and herbs without a page",
  "Three labels are displayed. 'The guide names this enzyme or drug for this herb' means the herb's page names the pair's enzyme or drug "
  "(called Mentioned in the tier rule). 'The guide's page for this herb does not name this enzyme or this drug' and 'The guide has no page for this "
  "herb' are both counted as Not mentioned in the tier rule. For herbs without a page in the guide, prior documentation is counted as not mentioned; "
  "this reflects coverage of the guide, not a negative finding. Four herbs (Parsley, Harmal, Clove, Anise) have no page, and Parsley's three Tier 1 "
  "pairs (with tacrolimus, sirolimus and everolimus) rest on this."),
 ("Dropped criterion: regional availability",
  "This criterion is not part of the tier rule and carries no criterion number. Reason: all six drugs are standard registered medicines, so it was "
  "expected to be positive for every pair and could not separate them. The survey of which Eastern Mediterranean countries publish free national "
  "medicines listings is kept as a separate finding, documentation-only, in the sheet '" + DROPPED_SHEET + "'."),
 ("Default rule for C3 Prior documentation (Rule 1)",
  "A pair is Mentioned if the herb's page names the pair's enzyme (in the Herb-Drug Interactions section, or in an Adverse Reactions case "
  "report) or names the pair's drug by name. A drug class does not count. Otherwise the pair is Not mentioned. A herb with no page is Not mentioned."),
 ("What 'Mentioned' means",
  "The herb's page names the pair's enzyme or drug. It does not mean the page documents that specific herb-drug interaction. Example: chamomile is "
  "'Mentioned' for tacrolimus because the page says chamomile inhibits CYP3A4 in vitro - not because it reports a chamomile-tacrolimus interaction."),
 ("C3 Prior documentation is a naming test, not a mechanism test",
  "C3 Prior documentation does not assess mechanism. It asks only whether the pair's enzyme or drug is named. Two examples show the consequence. "
  "(a) Green tea x S-warfarin is Not mentioned: green tea's page names no warfarin anywhere and names only the class 'anticoagulants / "
  "antiplatelets' (stated mechanism: vitamin K antagonism), and does not name CYP2C9. (b) Ginger x tacrolimus is Mentioned, on drug name alone: "
  "the page states that ginger increased tacrolimus plasma levels, gives no mechanism, and ginger names no CYP enzyme at all. So 'Mentioned' "
  "can rest on a non-CYP or unstated mechanism."),
 ("Prior documentation basis (column)",
  "One of: 'enzyme named - Herb-Drug Interactions'; 'enzyme named - case report only'; 'drug named'; 'enzyme not named'; 'no monograph' (the herb has "
  "no page). Turmeric x CYP2C9 pairs are 'enzyme named - case report only': CYP2C9 appears only in an Adverse Reactions case report (paclitaxel), "
  "not in the Herb-Drug Interactions list."),
 ("Scope of 'named'",
  "Enzymes and drugs are read from the Herb-Drug Interactions section and Adverse Reactions case reports. Each page also has a patient-facing summary "
  "at the top that can name drugs; that is outside the scope used here (decided 2026-10-29: the scope was fixed before it was known which pair the "
  "choice would move, so changing it afterwards would be post hoc). It affects one pair: the ginger page's summary names warfarin ('Do Not Take if: "
  "You are taking warfarin or other blood thinners'), so counting it would make ginger x S-warfarin Mentioned and move it from Tier 2 to Tier 3. "
  "Tier 1 is unaffected."),
 ("Which drug column is used",
  "data/c3_mskcc_lookup.csv has two drug columns. Drugs_named_all lists every drug named in the herb's Herb-Drug Interactions section. "
  "Drugs_named_among_study_drugs lists only those of the six study drugs (the drugs in the 29 pairs) that the page names. This sheet uses only "
  "Drugs_named_among_study_drugs; for the six study drugs the two columns are identical (checked in the notebook), and section 5.5 uses "
  "Drugs_named_all for the pairs outside the 29."),
 ("Cinnamon", f"The {GUIDE} lists Cinnamomum zeylanicum (with C. aromaticum, C. loureiroi, C. burmannii), a synonym of C. verum. The page covers "
              "several species, not C. verum specifically."),
 ("Novelty caveat", f"'Not mentioned' means 'not named on the herb's page', not 'unreported'. {n_no_mono} of 29 pairs are Not mentioned only because the "
                    f"herb (anise, clove, harmal, parsley) has no page in the {GUIDE}, which reflects the guide's coverage and is not evidence of novelty."),
 ("Not part of the tier", "The maximum predicted probability (the model's highest probability, across the herb's constituents, that one of them "
                          "inhibits the enzyme) and the independent evidence status (Supported, Untested or Contradicted, compared with independent "
                          "laboratory measurements held in ChEMBL) are shown beside the tier but are not in the tier rule. "
                          f"Independent evidence status of the {len(t1_rows)} Tier 1 pairs: {t1c}. Read the tier together with the independent evidence status."),
 ("Sensitivity of C3 Prior documentation to the rule",
  "See the sheet 'C3 sensitivity' (how the tier counts change with the rule; also written to outputs/tableS12_prior_documentation_rule_sensitivity.csv): Rule 2 (strict) ignores "
  "enzymes named only in a case report; Rule 3 (loose) lets a drug class name its member drugs. Tier 1 / Tier 2 / Tier 3 counts: " + rule_counts + ". "
  "The Tier 1 set is identical under all three rules."),
 ("Independent check", f"A second, independent lookup of the {GUIDE} (data/STEP_8c_c3_independent_pairs.csv) is compared pair by pair in "
                       f"outputs/tableS11_prior_documentation_agreement.csv: {n_agree} of 29 agree."),
 ("Sources for C3 Prior documentation", f"These support the choice of the {GUIDE} as a reference source. They do not validate the Prior "
                                        "documentation rule. All three were checked on PubMed on 2026-10-29."),
 ("Citation 1", "Sansevere ME, White JD. Quality Assessment of Online Complementary and Alternative Medicine Information Resources Relevant to Cancer. "
                "Integrative Cancer Therapies. 2021;20:15347354211066081. Digital object identifier 10.1177/15347354211066081; PubMed identifier 34923872."),
 ("Citation 2", "Rizvi RF, Vasilakes J, Adam TJ, Melton GB, Bishop JR, Bian J, Tao C, Zhang R. iDISK: the integrated DIetary Supplements Knowledge base. "
                "Journal of the American Medical Informatics Association. 2020;27(4):539-548. Digital object identifier 10.1093/jamia/ocz216; PubMed identifier 32068839."),
 ("Citation 3", "Hou Y-N, Deng G, Mao JJ. Practical Application of 'About Herbs' Website: Herbs and Dietary Supplement Use in Oncology Settings. "
                "Cancer Journal. 2019;25(5):357-366. Digital object identifier 10.1097/PPO.0000000000000403; PubMed identifier 31567464."),
 ("Sources for the tier rule", "Examples of category-based grading of reported interactions, and the FDA definitions behind the Substrate sensitivity "
                               "labels. Existence and bibliographic details of Citations 4 to 6 were checked on PubMed on 2026-10-29."),
 ("Citation 4", "De Smet PAGM. Clinical risk management of herb-drug interactions. British Journal of Clinical Pharmacology. 2007;63(3):258-267. "
                "Digital object identifier 10.1111/j.1365-2125.2006.02797.x; PubMed identifier 17116126. Its evidence-quality and seriousness codes are "
                "described here as applied in Citation 5; the full text of this paper could not be retrieved when checked."),
 ("Citation 5", "Pochet S, Lechon A-S, Lescrainier C, De Vriese C, Mathieu V, Hamdani J, Souard F. Herb-anticancer drug interactions in real life based on "
                "VigiBase, the WHO global database. Scientific Reports. 2022;12(1):14178. Digital object identifier 10.1038/s41598-022-17704-z; PubMed "
                "identifier 35986023."),
 ("Citation 6", "Cnudde A, Allely C, Biset N, Champy P, Fouilhe N, Huret F, Lawson S, Mercan A, Noukela Noumi DP, Michalet S, Montis A, Pochet S, Schils A, "
                "Tangeten C, Tod M, Van Antwerpen P, Vervacke A, Souard F. BABINE: An original and user-friendly scale for the simple and quick management of "
                "herb-drug interactions in clinical practice. BMC Complementary Medicine and Therapies. 2024;24(1):414. Digital object identifier "
                "10.1186/s12906-024-04706-9; PubMed identifier 39695590."),
 ("Citation 7", "Food and Drug Administration. Drug Development and Drug Interactions | Table of Substrates, Inhibitors and Inducers (notes to Table 2-1: "
                "sensitive substrates show an increase in exposure of 5-fold or more with strong index inhibitors; moderately sensitive substrates 2-fold to "
                "less than 5-fold). Page content current as of 2023-06-05; read 2026-10-29. "
                "https://www.fda.gov/drugs/drug-interactions-labeling/drug-development-and-drug-interactions-table-substrates-inhibitors-and-inducers"),
]
method_sheet = pd.DataFrame(METHOD, columns=["Item", "Text"])

# ------------------------------------------------------------------ Glossary sheet ----------------
GLOSSARY = [
 ("C1", "C1 Substrate sensitivity", "Label 1 of 3. How strongly the drug depends on the pair's CYP enzyme: Sensitive (≥5-fold) or Moderate (2 to 5-fold)."),
 ("C2", "C2 Narrow therapeutic index", "Label 2 of 3. Yes or No: whether the drug has a narrow therapeutic index. Yes for all six drugs, so it is shown but is not in the tier rule."),
 ("C3", "C3 Prior documentation", f"Label 3 of 3. Mentioned if the herb's page in the {GUIDE} already names the pair's enzyme or drug; Not mentioned if not."),
 ("Dropped criterion: regional availability", "Dropped criterion: regional availability", "A former criterion, removed because every drug was expected to give the same answer. Its country survey is kept in a documentation-only sheet."),
 ("CYP", "Cytochrome P450", "A family of liver enzymes that break down many medicines. CYP1A2, CYP2C9 and CYP3A4 are individual members of the family; their names are kept as they are."),
 (GUIDE, GUIDE, "The 'About Herbs' website of a New York cancer centre, with a page for each herb. It is the reference used for prior documentation."),
 ("FDA", "Food and Drug Administration", "The United States medicines regulator. Its published list classes drugs as sensitive or moderate substrates of each CYP enzyme."),
 ("IC50", "Half-maximal inhibitory concentration", "The concentration of a compound that cuts an enzyme's activity by half; the usual measure of how strongly it inhibits. The independent evidence status is built from IC50 values (or Ki where no IC50 exists)."),
 ("Ki", "Inhibition constant", "A second measure of how tightly an inhibitor binds an enzyme. Used only when no IC50 was available."),
 ("ChEMBL", "ChEMBL database", "A public database of measured drug-target activities. It supplies the independent laboratory measurements."),
 ("NSAIDs", "Nonsteroidal anti-inflammatory drugs", "Painkillers such as ibuprofen. Listed as a drug class in some pages; a class does not count as naming a drug."),
 ("BABINE", "Boosting Analysis of Bibliography for herb-drug INteraction Evaluation", "A published scale for grading reported herb-drug interactions (Citation 6 on the Method sheet)."),
 ("VigiBase", "VigiBase", "The World Health Organization's global database of suspected adverse drug reaction reports (Citation 5 on the Method sheet)."),
 ("Sensitive (≥5-fold)", "Sensitive substrate", "The drug's exposure rises 5-fold or more when a strong inhibitor blocks its enzyme (FDA definition, Citation 7)."),
 ("Moderate (2 to 5-fold)", "Moderate substrate", "The drug's exposure rises 2-fold to less than 5-fold when a strong inhibitor blocks its enzyme (FDA definition, Citation 7)."),
 ("Fold", "Times higher", "5-fold means five times higher."),
 ("Tier 1 - novel priority", "Sensitive AND Not mentioned", "Highest priority for review: a sensitive substrate whose enzyme or drug is not named on the herb's page."),
 ("Tier 2 - secondary", "Exactly one of: Sensitive; Not mentioned", "Second group: Sensitive and Mentioned, or Moderate and Not mentioned."),
 ("Tier 3 - low", "Neither Sensitive nor Not mentioned", "Lowest group: Moderate and Mentioned. The tier exists only in the spreadsheet."),
 (pdoc.LABEL_NAMES, "Displayed label (Mentioned in the tier rule)", "The herb's page names the pair's enzyme or drug. It does not mean the page documents that specific interaction."),
 (pdoc.LABEL_NO_MATCH, "Displayed label (Not mentioned in the tier rule)", "The herb has a page, but it names neither the pair's enzyme nor its drug."),
 (pdoc.LABEL_NO_PAGE, "Displayed label (Not mentioned in the tier rule)", f"The herb has no page in the {GUIDE} (Parsley, Harmal, Clove, Anise). This reflects coverage of the guide, not a negative finding."),
 ("Mentioned", "C3 Prior documentation = Mentioned", "Tier-rule term: the first displayed label."),
 ("Not mentioned", "C3 Prior documentation = Not mentioned", "Tier-rule term: the second or third displayed label."),
 ("Supported", "Independent evidence status", "An independent laboratory measurement agrees that the herb constituent inhibits the enzyme."),
 ("Untested", "Independent evidence status", "No independent laboratory measurement exists for that constituent and enzyme. This is absence of evidence, not evidence against."),
 ("Contradicted", "Independent evidence status", "An independent laboratory measurement found the constituent does not inhibit the enzyme (weaker than 10 micromolar), although the model predicted it would."),
 ("Rule 1 / Rule 2 / Rule 3", "Default / strict / loose", "Three ways of deciding whether a pair is Mentioned, used to test how much the tiers depend on the rule."),
 ("Study drugs", "The six drugs in the 29 pairs", "Theophylline, tacrolimus, sirolimus, everolimus, phenytoin and S-warfarin."),
 ("Monograph", "Monograph", f"A herb's own page in the {GUIDE}."),
 ("Case report", "Case report", "A published account of what happened to a single patient."),
 ("Maximum predicted probability", "Maximum predicted probability", "The model's highest probability, across the herb's constituents, that a constituent inhibits the enzyme. Shown for reading alongside the tier, not part of the rule."),
]
glossary_sheet = pd.DataFrame(GLOSSARY, columns=["Term", "Full form", "Plain-language meaning"])

# ------------------------------------------------------------------ checks on the finished sheets ---
# Everything the workbook will show, except the Glossary itself (which is where terms are defined).
FRAMES = {"Review": review_sheet, "Method": method_sheet, "C3 sensitivity": None,
          "C3 evidence": evidence_sheet, DROPPED_SHEET: dropped_sheet}
sens_frames = [sens_summary, sens_differs, sens_wide]
texts = [SENS_TITLE, DROPPED_NOTE, "Pairs whose tier differs from Rule 1", "Every pair under every rule"]
cells = []
for name, frame in FRAMES.items():
    for f in (sens_frames if frame is None else [frame]):
        texts += [str(c) for c in f.columns]
        cells += [str(v) for v in f.to_numpy().ravel()]
workbook_text = "\n".join(texts + cells)
glossary_text = "\n".join(str(v) for v in glossary_sheet.to_numpy().ravel())

# ASSERTION 11 - every abbreviation shown anywhere in the workbook is defined in the Glossary
ABBREVIATION = re.compile(r"\b[A-Z]{2,}[0-9A-Za-z]*\b")
ENGLISH_IN_CAPITALS = {"ONLY", "NOT", "AND", "SCOPE", "NOTE", "CLASS"}   # emphasis words that the lookup file writes in capitals
# The citation rows are exempt from THIS scan only: they hold authors' initials (ME, JD, RF ...), a published
# title's own capitalisation, the WHO and a digital object identifier. Assertions 12-14 below still scan them.
abbreviation_text = workbook_text
for citation in method_sheet.loc[method_sheet["Item"].str.startswith("Citation"), "Text"]:
    abbreviation_text = abbreviation_text.replace(citation, "")
used = {t for t in ABBREVIATION.findall(abbreviation_text)
        if t not in ENGLISH_IN_CAPITALS and not re.fullmatch(r"CYP\d[0-9A-Z]*", t)}   # CYP enzyme names stay as they are
defined = {t for term in glossary_sheet["Term"] for t in ABBREVIATION.findall(term)}
assert used <= defined, f"abbreviations used in the workbook but missing from the Glossary: {sorted(used - defined)}"

# ASSERTION 12 - no one-letter yes/no cell, and no leftover old criterion label anywhere in the workbook.
OLD_CRITERION_NUMBER = "4"
assert not set("YN") & set(cells), "a cell holds a one-letter yes/no abbreviation"
assert not re.search(rf"(?<![A-Za-z0-9])[Cc]{OLD_CRITERION_NUMBER}(?![A-Za-z0-9])|[Cc]{OLD_CRITERION_NUMBER}_", workbook_text), \
    "an old criterion label is left in the workbook"

# ASSERTION 13 - every C1 / C2 / C3 shown has its name beside it. The only exception is a reference to one of
criterion_text = workbook_text + "\n" + "\n".join(str(v) for v in glossary_sheet[["Full form", "Plain-language meaning"]].to_numpy().ravel())
for sheet_name in ("C3 sensitivity", "C3 evidence"):
    criterion_text = criterion_text.replace(f"'{sheet_name}'", "")
for m in re.finditer(r"\bC([123])\b(?!_)", criterion_text):
    tail = criterion_text[m.end():m.end() + 40]
    assert tail.startswith(" " + CRITERIA_NAMES[m.group(1)]), \
        f"C{m.group(1)} without its name: ...{criterion_text[max(0, m.start() - 25):m.end() + 40]!r}"

# ASSERTION 14 - the retired point scores and the short name of the guide appear nowhere that is displayed
assert "MSKCC" not in workbook_text + glossary_text, "the abbreviation for the Memorial Sloan Kettering guide is displayed"
for frame in [review_sheet, sens_summary, sens_differs, sens_wide]:
    for column in frame.columns:
        assert not re.search(r"(?i)\bscore|\btotal\b|\bpoints?\b", str(column)), f"a point-score header is left: {column!r}"

assert set(review_sheet.iloc[:, 8]) <= set(pdoc.DISPLAY_LABELS), "unexpected Prior documentation label in the Review sheet"
print(f"PASS  assertions 11-14: {len(used)} abbreviations all defined in the Glossary; no one-letter yes/no cells; no old criterion label; "
      "every criterion number carries its name; no point-score header; no short name for the guide")

# ------------------------------------------------------------------ write -------------------------
out_path = OUT / "tableS13_pharmacist_review_sheet.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl", mode="w") as xw:
    review_sheet.to_excel(xw, sheet_name="Review", index=False)
    method_sheet.to_excel(xw, sheet_name="Method", index=False)
    sens_summary.to_excel(xw, sheet_name="C3 sensitivity", index=False, startrow=1)
    sens_differs.to_excel(xw, sheet_name="C3 sensitivity", index=False, startrow=len(sens_summary) + 5)
    sens_wide.to_excel(xw, sheet_name="C3 sensitivity", index=False, startrow=len(sens_summary) + len(sens_differs) + 9)
    evidence_sheet.to_excel(xw, sheet_name="C3 evidence", index=False)
    dropped_sheet.to_excel(xw, sheet_name=DROPPED_SHEET, index=False, startrow=2)
    glossary_sheet.to_excel(xw, sheet_name="Glossary", index=False)

    from openpyxl.styles import Alignment, Font, PatternFill
    bold, fill = Font(bold=True), PatternFill("solid", fgColor="DDE6F0")
    sens = xw.sheets["C3 sensitivity"]
    sens["A1"] = SENS_TITLE
    sens.cell(row=len(sens_summary) + 4, column=1, value="Pairs whose tier differs from Rule 1")
    sens.cell(row=len(sens_summary) + len(sens_differs) + 8, column=1, value="Every pair under every rule")
    xw.sheets[DROPPED_SHEET]["A1"] = DROPPED_NOTE
    table_sheets = ("Review", "Method", "C3 evidence", "Glossary")
    for name, ws in xw.sheets.items():
        for cell in ws[1] if name in table_sheets else []:
            cell.font, cell.fill = bold, fill
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        for title_cell in ("A1",) if name in ("C3 sensitivity", DROPPED_SHEET) else ():
            ws[title_cell].font = bold
        ws.freeze_panes = "A2" if name in table_sheets else None
        for col in ws.columns:
            width = min(max((len(str(c.value)) for c in col[1:] if c.value is not None), default=8) + 2, 60)
            ws.column_dimensions[col[0].column_letter].width = max(width, 14)
        if name in ("Method", "C3 evidence", "Glossary", "Review"):
            for row in ws.iter_rows(min_row=2):
                for c in row:
                    c.alignment = Alignment(wrap_text=True, vertical="top")
    xw.sheets["Method"].column_dimensions["A"].width = 42
    xw.sheets["Method"].column_dimensions["B"].width = 120
    xw.sheets["Glossary"].column_dimensions["A"].width = 34
    xw.sheets["Glossary"].column_dimensions["B"].width = 38
    xw.sheets["Glossary"].column_dimensions["C"].width = 110
    xw.sheets["Review"].row_dimensions[1].height = 75

print("wrote", out_path.relative_to(PROJECT), f"({out_path.stat().st_size} bytes)")
print("sheets:", pd.ExcelFile(out_path).sheet_names)
print("\nReview sheet headers (internal name -> header shown):")
for old, new in REVIEW_COLUMNS.items():
    print(f"  {old:28s} -> {new}")

In [ ]:
print("=" * 78)
print("REVIEW SHEET - RESULT")
print("=" * 78)
print(f"\nPrior documentation under the default rule : Mentioned {n_doc}, Not mentioned {n_not}")
print("\nBasis of the 29 Prior documentation values :")
print(r1["C3_basis"].value_counts().to_string())
print("\nTier counts under all three rules (rule-based tier, no points):")
print(summary.to_string(index=False))
print(f"\nAgreement with the independent lookup: {n_agree} of 29 agree, {n_dis} disagree")
if n_dis:
    print(agree[~agree['Agree']].to_string(index=False))
print("\nAll assertion groups passed (a failed assertion stops this notebook with an error).")

### 5.5 Pair table

One row per herb-drug-enzyme combination with the prior-documentation, lab-comparison and drug-class labels, plus the table of combinations
without a pair and the reason.

In [ ]:
import re
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

import prior_documentation as pdoc

pairs = pd.read_csv(OUT / "table5_herb_drug_pairs.csv")
priority = pd.read_csv(OUT / "table6_priority_pairs.csv")
constituents = pd.read_csv(OUT / "table4_herb_constituent_predictions.csv")
lookup = pd.read_csv(DATA / "c3_mskcc_lookup.csv", keep_default_na=False)      # blank stays "", "None"/"NA" never become NaN
LK = lookup.set_index("Herb")
KEY = ["Herb", "Drug", "Enzyme"]

# ASSERTION 1 (first part) - the table this notebook is built on
assert len(pairs) == 423 and pairs["Herb"].nunique() == 12 and pairs["Drug"].nunique() == 97, \
    (len(pairs), pairs["Herb"].nunique(), pairs["Drug"].nunique())
assert not pairs.duplicated(KEY).any(), "duplicate (Herb, Drug, Enzyme) keys in table5_herb_drug_pairs.csv"
assert set(pairs["Herb"]) == set(lookup["Herb"]), "the lookup does not cover exactly the 12 herbs"
print(f"OK  {len(pairs)} pairs, {pairs['Herb'].nunique()} herbs, {pairs['Drug'].nunique()} drugs, no duplicate keys")

In [ ]:
SUBSTRATE_LABEL = {"sensitive": "Sensitive (≥5-fold)", "moderate": "Moderate (2 to 5-fold)"}
out = pairs[KEY + ["Max probability", "Driver", "Flagged constituents", "Measured uM", "Measured assay"]].copy()
out["Substrate sensitivity"] = pairs["FDA_substrate_class"].map(SUBSTRATE_LABEL).fillna("Not classified")
out["Narrow therapeutic index"] = pairs["NTI"].map({True: "Yes", False: "No"})
assert out["Narrow therapeutic index"].notna().all(), "a pair has no narrow-therapeutic-index value"

not_classified = out["Substrate sensitivity"] == "Not classified"
print("Substrate sensitivity:", out["Substrate sensitivity"].value_counts().to_dict())
print(f"Not classified: {int(not_classified.sum())} pairs, {out.loc[not_classified, 'Drug'].nunique()} drugs")
print("Narrow therapeutic index:", out["Narrow therapeutic index"].value_counts().to_dict())

In [ ]:
pdoc.self_test_token_in()          # the matcher's positive AND negative self-tests (CYP2C19 must not match 2C9, ...)

basis = [pdoc.basis(LK.loc[h], d, e, 1, "Drugs_named_all") for h, d, e in zip(out.Herb, out.Drug, out.Enzyme)]
out["_matcher_basis"] = basis
out["Prior documentation"] = [pdoc.display_label(b) for b in basis]

basis_label, matched_term = [], []
for herb, drug, enzyme in zip(out.Herb, out.Drug, out.Enzyme):
    rec = LK.loc[herb]
    if rec["In_MSKCC"] == "No":
        basis_label.append("no_page"); matched_term.append(""); continue
    enz = [e for e in pdoc.parts(rec["Enzymes_named"]) if pdoc.code_of(e) == pdoc.code_of(enzyme)]
    drg = [d for d in pdoc.parts(rec["Drugs_named_all"]) if pdoc.norm_drug(d) == pdoc.norm_drug(drug)]
    basis_label.append("both" if enz and drg else "enzyme" if enz else "drug" if drg else "none")
    matched_term.append("; ".join(x for x in (enz[0] if enz else "", drg[0] if drg else "") if x))
out["Prior documentation basis"], out["Matched term"] = basis_label, matched_term

# the explicit basis agrees with the matcher on every one of the 423 pairs
agree_matcher = [(b in ("enzyme", "drug", "both")) == pdoc.is_mentioned(m) and (b == "no_page") == (m == "no monograph")
                 for b, m in zip(basis_label, basis)]
assert all(agree_matcher), "the explicit basis disagrees with the matcher"

study = out.merge(priority[KEY], on=KEY, how="inner")
for h, d, e, b in zip(study.Herb, study.Drug, study.Enzyme, study["_matcher_basis"]):
    assert b == pdoc.basis(LK.loc[h], d, e, 1, "Drugs_named_among_study_drugs"), f"{h} x {d}: the two drug columns disagree"

unsupported = [(h, d, e) for h, d, e, b in zip(out.Herb, out.Drug, out.Enzyme, basis_label)
               if b in ("enzyme", "both") and pdoc.code_of(e) not in {pdoc.code_of(x) for x in pdoc.parts(LK.loc[h, "Enzymes_named"])}]
assert not unsupported, f"enzyme matches whose enzyme is not in Enzymes_named: {unsupported[:3]}"
print("Prior documentation:", out["Prior documentation"].value_counts().to_dict())
print("Basis:", out["Prior documentation basis"].value_counts().to_dict())

In [ ]:
assert CHEMBL_CUTOFF_NM == 10_000 and FLAG_THRESHOLD == 0.5, "cut-offs changed"
print("\nSo: Supported = measured median <= 10,000 nM (10 micromolar) AND model probability >= 0.5;")
print("    Contradicted = measured median > 10,000 nM (weaker than 10 micromolar) but model probability >= 0.5;")
print("    Untested = no independent measurement for that constituent and enzyme.")

SENTENCE = {"Supported": "A published lab test agrees with the model.",
            "Contradicted": "A published lab test disagrees with the model.",
            "Untested": "No published lab test found."}
out["Reference check"] = pairs["Evidence status"]
assert set(out["Reference check"]) <= set(SENTENCE), f"unexpected Evidence status: {set(out['Reference check']) - set(SENTENCE)}"
out["Reference check sentence"] = out["Reference check"].map(SENTENCE)

# the printed cut-off really is what the stored values show
meas = pd.to_numeric(pairs["Measured uM"], errors="coerce")
assert (meas[out["Reference check"] == "Supported"] <= 10).all() and (meas[out["Reference check"] == "Contradicted"] > 10).all(), \
    "stored Measured uM disagrees with the 10 micromolar cut-off"
print("\nchecked: every Supported pair has Measured uM <= 10 and every Contradicted pair has Measured uM > 10")

In [ ]:
out["Assessable"] = pairs["Assessable"]
assert out["Assessable"].between(2, 5).all(), "Assessable outside 2 to 5"
wide_dom = constituents.set_index(["Herb", "Compound"])
for herb, driver, enzyme, assessable in zip(out.Herb, out.Driver, out.Enzyme, out.Assessable):
    assert bool(wide_dom.loc[(herb, driver), f"{enzyme}_in_domain"]), f"{herb}: driver {driver} is out of domain for {enzyme}"
    assert assessable == int(constituents.loc[constituents.Herb == herb, f"{enzyme}_in_domain"].astype(bool).sum()),         f"{herb} {enzyme}: Assessable is not the number of in-domain constituents"
print("Assessable:", out["Assessable"].value_counts().sort_index().to_dict(), "| every driver is in domain for its enzyme (so the flag was constant)")

In [ ]:
priority_keys = set(map(tuple, priority[KEY].to_numpy()))
out["Priority pair"] = [tuple(k) in priority_keys for k in out[KEY].to_numpy()]

COLUMNS = ["Herb", "Drug", "Enzyme", "Max probability", "Driver", "Flagged constituents", "Assessable",
           "Substrate sensitivity", "Narrow therapeutic index",
           "Prior documentation", "Prior documentation basis", "Matched term",
           "Reference check", "Reference check sentence", "Measured uM", "Measured assay",
           "Priority pair"]
final = out[COLUMNS].copy()
LABELS = ["Substrate sensitivity", "Narrow therapeutic index", "Prior documentation", "Prior documentation basis",
          "Reference check", "Reference check sentence", "Assessable"]
assert set(final["Prior documentation"]) <= set(pdoc.DISPLAY_LABELS)
assert set(final["Prior documentation basis"]) <= {"enzyme", "drug", "both", "none", "no_page"}
assert ((final["Matched term"] != "") == final["Prior documentation basis"].isin(["enzyme", "drug", "both"])).all()

# ASSERTION 1 - shape
assert len(final) == 423 and final["Herb"].nunique() == 12 and final["Drug"].nunique() == 97, "wrong shape"
assert not final.duplicated(KEY).any(), "duplicate keys"

# ASSERTION 2 - no null in any displayed label column (nor in the identifying columns)
for column in LABELS + ["Herb", "Drug", "Enzyme", "Driver", "Max probability", "Flagged constituents"]:
    assert final[column].notna().all() and (final[column].astype(str).str.strip() != "").all(), f"null or blank in {column}"

# ASSERTION 3 - exactly the 29 priority pairs are flagged, and they are the ones in table6_priority_pairs.csv
flagged = set(map(tuple, final.loc[final["Priority pair"], KEY].to_numpy()))
assert int(final["Priority pair"].sum()) == 29 and flagged == priority_keys, "Priority pair flags do not match table6_priority_pairs.csv"

# ASSERTION 4 - for those 29, Prior documentation equals section 5.4's value (29 of 29)
n15 = pd.read_csv(OUT / "tableS12_prior_documentation_rule_sensitivity.csv", keep_default_na=False)[KEY + ["C3_rule1"]]
n15_basis = pd.read_csv(OUT / "tableS11_prior_documentation_agreement.csv", keep_default_na=False)[KEY + ["C3_basis_notebook"]]
final_two_way = final["Prior documentation"].map(lambda v: pdoc.MENTIONED if v == pdoc.LABEL_NAMES else pdoc.NOT_MENTIONED)
cmp = (final.assign(**{"Prior documentation": final_two_way}).loc[final["Priority pair"], KEY + ["Prior documentation"]]
       .merge(n15, on=KEY, how="inner", validate="1:1").merge(n15_basis, on=KEY, how="inner", validate="1:1"))
assert len(cmp) == 29, "section 5.4's table does not cover the 29 pairs "
same = (cmp["Prior documentation"] == cmp["C3_rule1"])
no_page_matches = ((final.loc[final["Priority pair"]].merge(cmp[KEY], on=KEY)["Prior documentation"] == pdoc.LABEL_NO_PAGE).values
                   == (cmp["C3_basis_notebook"] == "no monograph").values)
assert same.all() and no_page_matches.all(), f"Prior documentation differs from section 5.4:\n{cmp[~(same & no_page_matches)]}"
print(f"ASSERTION 4: Prior documentation equals section 5.4 for {int(same.sum())} of 29 priority pairs")

# ASSERTION 5 - the Reference check counts recorded since 27 Oct 2026
rc = final["Reference check"].value_counts().to_dict()
assert (rc.get("Supported"), rc.get("Untested"), rc.get("Contradicted")) == (33, 308, 82) and sum(rc.values()) == 423, rc

# ASSERTION 6 - no displayed text column contains the forbidden words
FORBIDDEN = re.compile(r"\b(?:safe|safely|avoid|dose|dosing|dangerous|harmless|recommend|should)\b", re.I)
for column in final.select_dtypes(exclude=["number", "bool"]).columns:
    hit = final[column].astype(str).str.contains(FORBIDDEN)
    assert not hit.any(), f"forbidden word in column {column}: {final.loc[hit, column].unique()[:3]}"

# ASSERTION 7 - no tier or points field
assert not any(re.search(r"(?i)tier|point|score|total", c) for c in final.columns), "a tier or points column is present"

final.to_csv(OUT / "table9_pair_table.csv", index=False)
print("PASS  assertions 1-7;  wrote outputs/table9_pair_table.csv", final.shape)
print(final.head(3).T.to_string())

# ----------------------------------------------------------------------------------------------- 13.7
# (continues below in the next section)

In [ ]:
substrates_all = pd.read_csv(DATA / "cyp_drug_substrates.csv")
ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]
herbs12 = sorted(pairs["Herb"].unique())                      # the 12 herbs that have pair rows
herbs_all = sorted(constituents["Herb"].unique())              # all 17 herbs that have constituent predictions
assert len(substrates_all) == 99 and len(herbs12) == 12 and len(herbs_all) == 17 and set(herbs12) <= set(herbs_all)

# section 5.1's herb x enzyme summary, recomputed here
signal = {}
for herb in herbs_all:
    group = constituents[constituents.Herb == herb]
    for enzyme in ENZYMES:
        values = group[enzyme]
        n = int(values.notna().sum())
        signal[(herb, enzyme)] = dict(compounds=len(group), assessable=n,
                                      highest=float(values.max()) if n else np.nan, flagged=int((values >= 0.5).sum()))

present = set(map(tuple, pairs[KEY].to_numpy()))
rows, categories = [], []
for herb in herbs_all:
    for drug, enzyme in zip(substrates_all.Drug, substrates_all.Enzyme):
        if (herb, drug, enzyme) in present:
            continue
        s = signal[(herb, enzyme)]
        assert s["flagged"] == 0, f"{herb} x {drug} ({enzyme}) is absent although a constituent was flagged"
        if s["assessable"] == 0:
            categories.append("none assessable")
            reason = (f"No compound of this herb could be assessed for {enzyme}: none of its {s['compounds']} compounds fell inside the "
                      "model's applicability domain, so the model gave no prediction.")
        else:
            categories.append("assessed, none reached 0.5")
            reason = (f"The model assessed {s['assessable']} compound{'s' if s['assessable'] != 1 else ''} of this herb for {enzyme}; "
                      f"none reached the 0.5 flagging threshold (the highest probability was {s['highest']:.3f}), so no herb-drug pair was made for this herb and enzyme.")
        rows.append((herb, drug, enzyme, reason))
missing = pd.DataFrame(rows, columns=["Herb", "Drug", "Enzyme", "Reason"])

for herb, drug, enzyme in present:
    assert signal[(herb, enzyme)]["flagged"] > 0, f"{herb} x {drug} ({enzyme}) is present although nothing was flagged"
assert len(missing) + len(final) == 17 * 99 == 1683 and not missing.duplicated(KEY).any()
# no herb is left with nothing to show: every one of the 17 herbs has pair rows, missing rows, or both
for herb in herbs_all:
    assert (final.Herb == herb).any() or (missing.Herb == herb).any(), f"{herb} has zero pair rows and zero missing rows"
assert not set(map(tuple, missing[KEY].to_numpy())) & present
assert not missing["Reason"].str.contains(FORBIDDEN).any(), "forbidden word in a Reason"
missing.to_csv(OUT / "tableS1_absent_combinations.csv", index=False)

by_reason = pd.Series(categories).value_counts()
print("PASS  wrote outputs/tableS1_absent_combinations.csv", missing.shape)
print("\ncount per reason:\n" + by_reason.to_string())
print("\nherb x enzyme combinations behind them:", {k: int(v) for k, v in pd.Series(
    [("none assessable" if s["assessable"] == 0 else "assessed, none reached 0.5") for (h, e), s in signal.items() if s["flagged"] == 0]).value_counts().items()})
below = [s["highest"] for s in signal.values() if s["flagged"] == 0 and s["assessable"]]
print(f"highest probability among absent herb-enzyme combinations: {max(below):.3f} (threshold 0.5)")
dup_drugs = substrates_all[substrates_all.Drug.duplicated(keep=False)].sort_values(["Drug", "Enzyme"])
print("\ndrugs that appear with two enzymes (99 drug-enzyme rows vs 97 drugs):")
print(dup_drugs[["Drug", "Enzyme", "FDA_substrate_class"]].to_string(index=False))
print("\nper herb: pair rows / missing rows (the five herbs with no pair row appear only in the missing file):")
for herb in herbs_all:
    print(f"   {herb:42s} {int((final.Herb == herb).sum()):4d} / {int((missing.Herb == herb).sum()):4d}")

In [ ]:
print("=" * 78); print("PAIR TABLE - RESULT"); print("=" * 78)
print(f"\nRows {len(final)}, herbs {final.Herb.nunique()}, drugs {final.Drug.nunique()}")
print("Substrate sensitivity  :", final["Substrate sensitivity"].value_counts().to_dict(),
      f"| Not classified: {int(not_classified.sum())} pairs, {out.loc[not_classified, 'Drug'].nunique()} drugs")
print("Narrow therapeutic index:", final["Narrow therapeutic index"].value_counts().to_dict())
print("Prior documentation    :", final["Prior documentation"].value_counts().to_dict())
print("   of which the 29 priority pairs:", final.loc[final["Priority pair"], "Prior documentation"].value_counts().to_dict())
print("Reference check        :", final["Reference check"].value_counts().to_dict())
print("Prior documentation basis:", final["Prior documentation basis"].value_counts().to_dict())

family = [(h, e) for h, r in LK.iterrows() for e in pdoc.parts(r["Enzymes_named"]) if not re.fullmatch(r"(?i)cyp\s*\d[A-Z]\d+", e.strip())]
print("\nEnzyme names on the guide's pages that are a FAMILY, not one enzyme (they match no enzyme):", family)
for h, e in family:
    n = int(((final.Herb == h) & final.Enzyme.str.startswith(e.strip().upper().replace(" ", ""))).sum())
    print(f"   {h}: {n} pair(s) whose enzyme starts with {e.strip()}")

# 10 random pairs OUTSIDE the 29, each with the basis for its Prior documentation value
rng = np.random.default_rng(16)
outside = out[~out["Priority pair"]]
pick = outside.loc[rng.choice(outside.index, size=10, replace=False)]
print("\n10 random pairs outside the 29 (seed 16):")
for _, r in pick.iterrows():
    rec = LK.loc[r.Herb]
    print(f"\n* {r.Herb} x {r.Drug} ({r.Enzyme})  ->  {r['Prior documentation']}   [basis: {r['Prior documentation basis']}; matched term: {r['Matched term'] or '-'}]")
    if r["Prior documentation basis"] == "no_page":
        print("    no page for this herb among the 308 entries enumerated")
    else:
        print("    page names enzymes:", rec["Enzymes_named"] or "-", "| case-report-only:", rec["Enzymes_case_report_only"] or "-")
        print("    page names drugs  :", rec["Drugs_named_all"] or "-")
print("\nAll pairs matched on a DRUG name outside the 29 priority pairs:")
drug_rows = out[(out["Prior documentation basis"] == "drug") & ~out["Priority pair"]]
print(drug_rows[["Herb", "Drug", "Enzyme"]].to_string(index=False) if len(drug_rows) else "  none")

## 6. Herb-level scoring

The herb score used throughout is the maximum probability among assessable compounds. This section shows how the ranking
changes under the mean and the mean of the two highest compounds, how many herb-enzyme calls rest on a single compound, and
how the priority pairs depend on it.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


import mixture as mx

ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]
constituents = pd.read_csv(OUT / "table4_herb_constituent_predictions.csv")

print(f"{len(constituents)} constituents across {constituents['Herb'].nunique()} herbs")
print(f"flag threshold: {mx.FLAG_THRESHOLD}")
print(f"sensitivity rules: {', '.join(mx.SENSITIVITY_RULES)}")
print("\nnoisy-OR is deliberately excluded — see 10c.4")

In [ ]:
rows = []
for (herb, enzyme), values in (
    ((herb, enzyme), pd.to_numeric(group[enzyme], errors="coerce"))
    for herb, group in constituents.groupby("Herb") for enzyme in ENZYMES
):
    score = mx.herb_score(values)
    if score is None:
        rows.append({"Herb": herb, "Enzyme": enzyme, "Assessable": 0})
        continue
    rows.append({
        "Herb": herb, "Enzyme": enzyme,
        "Max": round(score["max"], 3),
        "Flagged": score["n_flagged"],
        "Assessable": score["n_assessable"],
        "Second": None if score["second"] is None else round(score["second"], 3),
        "Margin": mx.margin(score),
        "Single-constituent": mx.single_constituent(score),
        "Mean": round(score["mean"], 3),
        "Top2": round(score["top2"], 3),
    })

herb_scores = pd.DataFrame(rows)
herb_scores["Flagged"] = herb_scores["Flagged"].astype("Int64")
herb_scores.to_csv(OUT / "tableS8_herb_level_scores.csv", index=False)
print("wrote outputs/tableS8_herb_level_scores.csv\n")
herb_scores[herb_scores["Enzyme"] == "CYP1A2"].sort_values(
    "Max", ascending=False).head(12)

In [ ]:
flagged = herb_scores[(herb_scores["Max"] >= mx.FLAG_THRESHOLD)].copy()
flagged["Herb"] = flagged["Herb"].str.split(" (", regex=False).str[0]

print("Herb-enzyme pairs above the threshold, by how many constituents support them:\n")
print(flagged.sort_values(["Flagged", "Max"], ascending=[False, False])[
    ["Herb", "Enzyme", "Max", "Flagged", "Assessable", "Second", "Margin"]
].to_string(index=False))

single = flagged["Single-constituent"].sum()
print(f"\n{single} of {len(flagged)} flagged herb-enzyme pairs rest on ONE constituent.")

In [ ]:
pairs = pd.read_csv(OUT / "table5_herb_drug_pairs.csv")

# Tier 1 = narrow-therapeutic-index drugs. Built from herb_drug_pairs rather than read
# from table6_priority_pairs.csv so this cell does not depend on which columns section 5.1 or
# section 5.2 happened to leave in that file.
keep = ["Herb", "Drug", "Enzyme", "Driver", "Max probability"]
if "Evidence status" in pairs.columns:
    keep.append("Evidence status")
priority = pairs[pairs["Priority"].str.startswith("1")][keep].copy()

annotated = priority.merge(
    herb_scores[["Herb", "Enzyme", "Flagged", "Second", "Margin"]],
    on=["Herb", "Enzyme"], how="left")

n = len(annotated)
none_flagged = int((annotated["Flagged"] == 0).sum())
one_flagged = int((annotated["Flagged"] == 1).sum())
several = int((annotated["Flagged"] >= 2).sum())

print(f"Priority (Tier 1) pairs: {n}\n")
print("How many constituents of the herb reach the 0.70 flag on that enzyme:")
print(f"   none  : {none_flagged:2d}  (admitted to the pair table at >= 0.50; Tier 1 is set")
print(f"                  by the DRUG being narrow-therapeutic-index, not by signal strength)")
print(f"   one   : {one_flagged:2d}")
print(f"   2 or more: {several:2d}")
print(f"\nAt most ONE constituent reaches 0.70: {none_flagged + one_flagged} of {n} "
      f"({100 * (none_flagged + one_flagged) / n:.0f}%)")
print("Do NOT describe all of these as 'resting on one flagged constituent' —")
print(f"{none_flagged} of them have no constituent at the flag threshold at all.\n")

annotated["Herb"] = annotated["Herb"].str.split(" (", regex=False).str[0]
print("The most single-constituent-dependent priority pairs:")
print(annotated.nlargest(6, "Margin")[
    ["Herb", "Drug", "Enzyme", "Driver", "Max probability", "Second", "Margin"]
].to_string(index=False))

annotated.to_csv(OUT / "tableS10_priority_pairs_annotated.csv", index=False)
print("\nwrote outputs/tableS10_priority_pairs_annotated.csv")

In [ ]:
demo = []
for herb, group in constituents.groupby("Herb"):
    values = pd.to_numeric(group["CYP1A2"], errors="coerce").dropna()
    if values.empty:
        continue
    demo.append({"Herb": herb.split(" (")[0],
                 "max": round(values.max(), 3),
                 "noisy_or": round(1 - np.prod(1 - values), 3)})
demo = pd.DataFrame(demo).sort_values("noisy_or", ascending=False)
saturated = (demo["noisy_or"] >= 0.999).sum()
print(f"Herbs with noisy-OR >= 0.999 on CYP1A2: {saturated}")
print(f"Herbs with noisy-OR >= 0.9:            {(demo['noisy_or'] >= 0.9).sum()} "
      f"of {len(demo)}\n")
print(demo.head(8).to_string(index=False))
print("\nThis is why noisy-OR is excluded: it cannot discriminate.")

In [ ]:
sensitivity = []
for enzyme in ENZYMES:
    subset = herb_scores[(herb_scores["Enzyme"] == enzyme) &
                         herb_scores["Max"].notna()]
    if len(subset) < 4:
        continue
    row = {"Enzyme": enzyme, "Herbs": len(subset)}
    for rule, column in [("mean", "Mean"), ("top2", "Top2")]:
        row[f"rho vs {rule}"] = round(
            spearmanr(subset["Max"], subset[column]).statistic, 3)
    # Does the top-4 set survive the change of rule?
    top_max = set(subset.nlargest(4, "Max")["Herb"])
    row["Top-4 same (mean)"] = top_max == set(subset.nlargest(4, "Mean")["Herb"])
    row["Top-4 same (top2)"] = top_max == set(subset.nlargest(4, "Top2")["Herb"])
    sensitivity.append(row)

sensitivity = pd.DataFrame(sensitivity)
c1a2 = herb_scores[(herb_scores["Enzyme"] == "CYP1A2") & herb_scores["Max"].notna()]
print("CYP1A2 top four under each rule:")
for rule, column in [("max", "Max"), ("mean", "Mean"), ("top-2", "Top2")]:
    names = [h.split(" (", 1)[0] for h in c1a2.nlargest(4, column)["Herb"]]
    print(f"   {rule:6s} {names}")
print()
sensitivity.to_csv(OUT / "tableS9_mixture_rule_sensitivity.csv", index=False)
print("wrote outputs/tableS9_mixture_rule_sensitivity.csv\n")
sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

subset = herb_scores[(herb_scores["Enzyme"] == "CYP1A2") & herb_scores["Max"].notna()]
order = subset.sort_values("Max", ascending=False)
labels = order["Herb"].str.split(" (", regex=False).str[0]

axes[0].scatter(order["Max"], order["Mean"], label="mean", s=45)
axes[0].scatter(order["Max"], order["Top2"], label="top-2", s=45, marker="^")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("maximum (the rule we use)")
axes[0].set_ylabel("alternative rule")
axes[0].set_title("CYP1A2 — alternative rules vs the maximum")
axes[0].legend()
axes[0].grid(alpha=0.3)

# The two axes we actually report: maximum and number of flagged constituents.
axes[1].scatter(order["Max"], order["Flagged"], s=70, color="crimson")
for _, row in order.iterrows():
    axes[1].annotate(row["Herb"].split(" (")[0], (row["Max"], row["Flagged"]),
                     fontsize=7, xytext=(4, 3), textcoords="offset points")
axes[1].axvline(mx.FLAG_THRESHOLD, color="grey", linestyle="--", alpha=0.6)
axes[1].set_xlabel("maximum constituent probability")
axes[1].set_ylabel("number of flagged constituents")
axes[1].set_title("The two reported axes — CYP1A2")
axes[1].grid(alpha=0.3)

fig.tight_layout()
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig4_mixture_scoring.png", dpi=150)
plt.show()

## 7. Pharmacogenomic layer

### 7.1 Phenotype frequencies from published allele frequencies

Star-allele frequencies are converted to metaboliser phenotype frequencies under Hardy-Weinberg equilibrium (`src/pgx_convert.py`) and checked
against the phenotype frequencies the source papers report.

In [ ]:
import sys
from pathlib import Path

import pandas as pd


import pgx_convert as pgx

print("populations with published allele data:", len(pgx.SOURCES))
for name in pgx.SOURCES:
    print("  -", name, "->", ", ".join(pgx.SOURCES[name]))

In [ ]:
rows = []
for population, enzymes in pgx.SOURCES.items():
    for enzyme, alleles in enzymes.items():
        for allele, frequency in sorted(alleles.items()):
            rows.append({"Population": population, "Enzyme": enzyme,
                         "Allele": allele, "Frequency": frequency})

alleles_table = pd.DataFrame(rows)
print(f"{len(alleles_table)} published allele frequencies transcribed")
alleles_table.head(20)

In [ ]:
derived = pd.DataFrame([
    {"Population": r["population"], "Enzyme": r["enzyme"], **r["phenotypes"]}
    for r in pgx.derive_all()
]).astype(object).fillna("")

derived

In [ ]:
checks = [
    ("Saudi (Goljan 2022)",       "CYP3A5",  "PM", 77.8, "CYP3A5 non-expressers (PM) 77.8%"),
    ("Saudi (Goljan 2022)",       "CYP2C9",  "NM", 64.5, "CYP2C9 EM 64.5%"),
    ("Saudi (Goljan 2022)",       "CYP2C19", "NM", 38.3, "CYP2C19 EM 38.3%"),
    ("Turkish (Aynacioglu 1999)", "CYP2D6",  "PM", 1.49, "6 of 404 predicted PM"),
    ("Turkish (Aynacioglu 1999)", "CYP2C19", "PM", 1.0,  "4 of 404 predicted PM"),
]

validation = pd.DataFrame([
    {"Population": population, "Enzyme": enzyme, "Phenotype": phenotype,
     "Derived %": pgx.CONVERTERS[enzyme](pgx.SOURCES[population][enzyme])[phenotype],
     "Reported %": reported, "Reported as": note}
    for population, enzyme, phenotype, reported, note in checks
])
validation["Difference"] = (validation["Derived %"] - validation["Reported %"]).round(1)

print(f"largest disagreement: {validation['Difference'].abs().max()} percentage points\n")
validation

In [ ]:
cyp2d6 = derived[derived["Enzyme"] == "CYP2D6"][["Population", "PM", "IM", "NM", "UM"]]
print("Derived CYP2D6 — note the UM column:\n")
print(cyp2d6.to_string(index=False))

assert (cyp2d6["UM"] == 0.0).all(), "a derived UM is non-zero - re-read 9a.5"
print("\nConfirmed: every derived UM is 0.0% — do not quote these.")

In [ ]:
table = pd.read_csv(PROJECT / "data" / "pgx_phenotype_frequencies.csv")

print(f"{len(table)} rows\n")
print(table["Value_type"].value_counts().to_string(), "\n")
table[["Population", "Enzyme", "PM_percent", "IM_percent", "NM_percent",
       "RM_percent", "UM_percent", "Value_type"]]

In [ ]:
regions = ["Middle East (Koopmans 2021)", "Europe (Koopmans 2021)", "East Asia (Koopmans 2021)"]
comparison = derived[derived["Population"].isin(regions)].set_index(["Population", "Enzyme"])

print("Same source, same method, three regions:\n")
print(comparison.to_string())

me = pgx.cyp2d6_phenotypes(pgx.SOURCES["Middle East (Koopmans 2021)"]["CYP2D6"])
eu = pgx.cyp2d6_phenotypes(pgx.SOURCES["Europe (Koopmans 2021)"]["CYP2D6"])
print(f"\nCYP2D6 intermediate metabolisers: Middle East {me['IM']}%  vs  Europe {eu['IM']}%")
print("Driven by CYP2D6*41 at 0.25 — the highest of any region in Koopmans Table 2.")

In [ ]:
derived.to_csv("outputs/tableS17_pgx_derived_phenotypes.csv", index=False)
validation.to_csv("outputs/tableS18_pgx_derivation_check.csv", index=False)

### 7.2 Phenoconversion

A moderate inhibitor multiplies CYP2D6 activity by 0.5 in every genotype, so the relative effect is the same in every population;
populations differ in absolute post-inhibition capacity. This section shows the activity-score distributions before and after, the cumulative
curves, the instability of threshold-based conversion counts, and which flagged herb-drug pairs fall on the enzymes covered.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


import phenoconversion as pheno

MODERATE = 0.5          # CPIC multiplier for a moderate inhibitor (Crews 2021)
ENZYME = "CYP2D6"       # the enzyme with data for all four populations

POPULATIONS = pheno.populations_for(ENZYME)
SHORT = {p: p.split(" (")[0] for p in POPULATIONS}

print("populations:", ", ".join(SHORT.values()))
print("bands -> CPIC multiplier:")
for cutoff, name, multiplier in pheno.BANDS:
    print(f"   p >= {cutoff:.2f}  {name:18s} x{multiplier}")
print("\nNo herb is ever assigned the 'strong' (x0) multiplier.")

In [ ]:
rows = []
for population in POPULATIONS:
    before = pheno.activity_distribution(population, ENZYME)
    after = pheno.shift_distribution(population, ENZYME, MODERATE)
    mean_before = pheno.mean_activity(before)
    mean_after = pheno.mean_activity(after)
    rows.append({"Population": SHORT[population],
                 "Mean AS before": round(mean_before, 3),
                 "Mean AS after": round(mean_after, 3),
                 "Ratio": round(mean_after / mean_before, 3)})

invariance = pd.DataFrame(rows)
assert (invariance["Ratio"] == MODERATE).all(), "relative effect is NOT invariant"
print("Confirmed: the relative effect is identical in every population.\n")
invariance

In [ ]:
reserve = pd.DataFrame([
    {"Population": SHORT[p],
     "At full activity score (AS 2.0)": f"{100 * pheno.reserve(p, ENZYME):.1f}%"}
    for p in POPULATIONS
])
reserve

In [ ]:
GRID = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 1.0, 1.25, 1.5, 2.0]

def on_grid(distribution, label):
    """Put every distribution on one shared axis so the shift is actually visible."""
    return {"Population": label,
            **{f"AS {s}": round(100 * distribution.get(s, 0.0), 1) for s in GRID}}

before_table = pd.DataFrame([
    on_grid(pheno.activity_distribution(p, ENZYME), SHORT[p]) for p in POPULATIONS
])
after_table = pd.DataFrame([
    on_grid(pheno.shift_distribution(p, ENZYME, MODERATE), SHORT[p]) for p in POPULATIONS
])

print("BEFORE — baseline activity score (% of population)")
print(before_table.to_string(index=False))
print("\nAFTER a moderate inhibitor (x0.5) — same axis, so you can see the mass move left")
print(after_table.to_string(index=False))

print("\n% of each population left at AS >= 1.0 (i.e. still at least intermediate):")
for population in POPULATIONS:
    after = pheno.shift_distribution(population, ENZYME, MODERATE)
    retained = sum(f for s, f in after.items() if s >= 1.0)
    print(f"   {SHORT[population]:14s} {100 * retained:5.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (distribution_fn, title) in zip(axes, [
    (lambda p: pheno.activity_distribution(p, ENZYME), "Before"),
    (lambda p: pheno.shift_distribution(p, ENZYME, MODERATE), "After a moderate inhibitor"),
]):
    for population in POPULATIONS:
        distribution = distribution_fn(population)
        ax.plot(list(distribution), [100 * v for v in distribution.values()],
                marker="o", label=SHORT[population])
    ax.set_title(title)
    ax.set_xlabel("CYP2D6 activity score")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("% of population")
axes[0].legend()
fig.suptitle("Where each population sits, before and after — no threshold applied")
fig.tight_layout()

OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "figS5_pgx_activity_distributions.png", dpi=150)
plt.show()

In [ ]:
grid = [round(0.05 * i, 2) for i in range(0, 45)]

curves = {}
for population in POPULATIONS:
    curves[(SHORT[population], "before")] = pheno.cumulative_below(
        pheno.activity_distribution(population, ENZYME), grid)
    curves[(SHORT[population], "after")] = pheno.cumulative_below(
        pheno.shift_distribution(population, ENZYME, MODERATE), grid)

fig, ax = plt.subplots(figsize=(9, 5.5))
colours = dict(zip(SHORT.values(), plt.rcParams["axes.prop_cycle"].by_key()["color"]))
for (label, when), curve in curves.items():
    ax.plot(grid, [100 * v for v in curve.values()], color=colours[label],
            linestyle="--" if when == "before" else "-",
            label=f"{label} ({when})", alpha=0.85)
ax.set_xlabel("activity score threshold t")
ax.set_ylabel("% of population with activity score ≤ t")
ax.set_title("Cumulative activity score, before (dashed) and after (solid)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUT / "figS6_pgx_cumulative_curves.png", dpi=150)
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "t": grid,
    "Middle East after": [100 * curves[("Middle East", "after")][t] for t in grid],
    "Europe after": [100 * curves[("Europe", "after")][t] for t in grid],
})
comparison["Middle East higher?"] = comparison["Middle East after"] > comparison["Europe after"]

crossings = comparison["Middle East higher?"].ne(comparison["Middle East higher?"].shift()).sum() - 1
print(f"The two curves cross {crossings} time(s) across t = 0 to {grid[-1]}.")
print("Wherever they cross, the sign of the 'difference' depends on the threshold.\n")
comparison[comparison["t"] <= 1.5].to_string(index=False)

In [ ]:
sweep = []
for threshold in (0.25, 0.4, 0.5, 0.6, 0.75, 1.0, 1.25):
    for strict in (True, False):
        values = {SHORT[p]: pheno.converted_fraction(
            pheno.activity_distribution(p, ENZYME), MODERATE, threshold, strict=strict)
            for p in POPULATIONS}
        europe = values["Europe"]
        sweep.append({"threshold": threshold,
                      "rule": "AS_after < t" if strict else "AS_after <= t",
                      "Middle East %": round(100 * values["Middle East"], 1),
                      "Europe %": round(100 * europe, 1),
                      "ratio": round(values["Middle East"] / europe, 1) if europe else None})

sweep = pd.DataFrame(sweep)
ratios = sweep["ratio"].dropna()
print(f"ratio ranges from {ratios.min()}x to {ratios.max()}x")
print(f"direction flips: Middle East is higher in {(ratios > 1).sum()} of {len(ratios)} settings")
print("\nA single number drawn from this table is not a measurement.\n")
sweep

In [ ]:
saturation = pd.DataFrame([
    {"Population": SHORT[p],
     "Mean AS after moderate (x0.5)": round(
         pheno.mean_activity(pheno.shift_distribution(p, ENZYME, 0.5)), 3),
     "Mean AS after strong (x0)": round(
         pheno.mean_activity(pheno.shift_distribution(p, ENZYME, 0.0)), 3)}
    for p in POPULATIONS
])
print("Under a strong inhibitor every population converges on zero —")
print("population pharmacogenomics stops mattering entirely.\n")
saturation

In [ ]:
pairs = pd.read_csv(OUT / "table5_herb_drug_pairs.csv")
constituents = pd.read_csv(PROJECT / "data" / "herb_constituents.csv")

herb_table = pd.read_csv("outputs/table4_herb_constituent_predictions.csv")
assert len(constituents) == len(herb_table) == 71 and constituents["Compound"].nunique() == 69, "unexpected herb compound list"
assert set(pairs["Herb"]) <= set(constituents["Herb"])

covered = pairs[pairs["Enzyme"].isin(pheno.ENZYMES)]
flagged = covered[covered["Max probability"] >= 0.70]

print(f"herb-drug pairs in total:                     {len(pairs)}")
print(f"on CYP2D6 or CYP2C9 (this layer's scope):     {len(covered)}")
print(f"of those, flagged at >= 0.70:                 {len(flagged)}")

if flagged.empty:
    print("\n  NO flagged pair falls on an enzyme this layer can model.")
    print("    Every flagged finding sits on CYP1A2, which has no activity score.")
    print("    The phenoconversion layer is therefore a method demonstration:")
    print("    it adds no evidence to any herb-drug pair reported here.")
else:
    display(flagged[["Herb", "Enzyme", "Drug", "Driver", "Max probability", "Priority"]])

In [ ]:
invariance.to_csv("outputs/tableS19_phenoconversion_invariance.csv", index=False)
reserve.to_csv("outputs/tableS20_baseline_reserve.csv", index=False)
before_table.to_csv("outputs/tableS21_activity_before.csv", index=False)
after_table.to_csv("outputs/tableS22_activity_after_moderate.csv", index=False)
sweep.to_csv("outputs/tableS23_threshold_sweep.csv", index=False)

## 8. Comparison with an existing tool (not a validation)

DEEPCYPs (Ai et al., Front Pharmacol 2023;14:1099093) was trained on the same PubChem AID 1851 screening data as the models here, so
agreement between the two cannot show that either is correct. The comparison asks whether two architectures trained on the same data
agree when applied to herb compounds. The DEEPCYPs predictions were produced once with the authors' published checkpoints
(`scripts/run_deepcyps.py`; see the README) and are stored in `data/benchmark/deepcyps_predictions.csv`, which this section reads; DEEPCYPs does not need to be installed.

This is a comparison with another tool, not a validation: both tools learned from the same screening data, so agreement does not show that either is correct. The "outside the domain" group below uses the model's unmasked probability so that it has enough compounds to compare.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ENZYMES = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]
THRESHOLD = 0.70          # strong-signal threshold used for the pair tiers

theirs = pd.read_csv(PROJECT / "data" / "benchmark" / "deepcyps_predictions.csv")
ours = pd.read_csv(OUT / "table4_herb_constituent_predictions.csv")

print(f"DEEPCYPs predictions: {len(theirs)} constituents")
print(f"our predictions:      {len(ours)} constituents")

merged = ours.merge(theirs, on=["Herb", "Compound"], suffixes=("_ours", "_theirs"))
print(f"matched on Herb + Compound: {len(merged)}")
assert len(merged) == len(theirs), "constituents went missing in the merge"

In [ ]:
rows = []
paired = {}
for enzyme in ENZYMES:
    a = pd.to_numeric(merged[f"{enzyme}_ours"], errors="coerce")
    b = pd.to_numeric(merged[f"{enzyme}_theirs"], errors="coerce")
    both = a.notna() & b.notna()
    a, b = a[both], b[both]
    paired[enzyme] = (a, b)

    flag_a, flag_b = a >= THRESHOLD, b >= THRESHOLD
    rows.append({
        "Enzyme": enzyme,
        "Compared": int(both.sum()),
        "Spearman": round(spearmanr(a, b).statistic, 3),
        "Mean |diff|": round(float((a - b).abs().mean()), 3),
        "Flagged ours": int(flag_a.sum()),
        "Flagged theirs": int(flag_b.sum()),
        "Both flagged": int((flag_a & flag_b).sum()),
        "Binary agreement": round(float((flag_a == flag_b).mean()), 3),
    })

agreement = pd.DataFrame(rows)
agreement.to_csv(OUT / "tableS6_deepcyps_agreement.csv", index=False)
print("wrote outputs/tableS6_deepcyps_agreement.csv\n")
agreement

In [ ]:
fig, axes = plt.subplots(1, len(ENZYMES), figsize=(4 * len(ENZYMES), 4.2), sharex=True,
                         sharey=True)
for ax, enzyme in zip(axes, ENZYMES):
    a, b = paired[enzyme]
    ax.scatter(a, b, alpha=0.65, s=32)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.axhline(THRESHOLD, color="crimson", lw=0.8, alpha=0.5)
    ax.axvline(THRESHOLD, color="crimson", lw=0.8, alpha=0.5)
    rho = spearmanr(a, b).statistic
    ax.set_title(f"{enzyme}\nSpearman {rho:.2f}  (n={len(a)})")
    ax.set_xlabel("our model")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("DEEPCYPs")
fig.suptitle("Same training data, different architectures — agreement on 71 herbal constituents")
fig.tight_layout()

OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig3_deepcyps_scatter.png", dpi=150)
plt.show()

In [ ]:
# Domain flag per enzyme from section 2/3 (applicability domain of that enzyme's training set); our own probability is
# taken from the unmasked model output, so compounds outside the domain can be compared as well.
raw = pd.read_csv("outputs/tableS16_herb_raw_probabilities.csv")
domain = ours[["Herb", "Compound"] + [f"{e}_in_domain" for e in ENZYMES]]
with_domain = merged[["Herb", "Compound"] + [f"{e}_theirs" for e in ENZYMES]].merge(
    raw.rename(columns={e: f"{e}_raw" for e in ENZYMES}), on=["Herb", "Compound"]).merge(domain, on=["Herb", "Compound"])

rows = []
for enzyme in ENZYMES:
    a = pd.to_numeric(with_domain[f"{enzyme}_raw"], errors="coerce")
    b = pd.to_numeric(with_domain[f"{enzyme}_theirs"], errors="coerce")
    inside = with_domain[f"{enzyme}_in_domain"].astype(bool)
    for label, mask in [("in domain", inside), ("out of domain", ~inside)]:
        sel = mask & a.notna() & b.notna()
        if sel.sum() < 3:
            rows.append({"Enzyme": enzyme, "Group": label, "n": int(sel.sum()), "Spearman": None, "Mean |diff|": None})
            continue
        rows.append({"Enzyme": enzyme, "Group": label, "n": int(sel.sum()),
                     "Spearman": round(spearmanr(a[sel], b[sel]).statistic, 3),
                     "Mean |diff|": round(float((a[sel] - b[sel]).abs().mean()), 3)})

by_domain = pd.DataFrame(rows)
by_domain.to_csv("outputs/tableS24_deepcyps_agreement_by_domain.csv", index=False)
by_domain

In [ ]:
flagged = []
for enzyme in ENZYMES:
    a = pd.to_numeric(merged[f"{enzyme}_ours"], errors="coerce")
    b = pd.to_numeric(merged[f"{enzyme}_theirs"], errors="coerce")
    for index in merged.index[(a >= THRESHOLD).fillna(False)]:
        flagged.append({
            "Herb": merged.loc[index, "Herb"].split(" (")[0],
            "Compound": merged.loc[index, "Compound"],
            "Enzyme": enzyme,
            "Ours": round(a[index], 3),
            "DEEPCYPs": round(b[index], 3),
            "They flag too": bool(b[index] >= THRESHOLD),
        })

flagged = pd.DataFrame(flagged).sort_values("Ours", ascending=False)
confirmed = flagged["They flag too"].sum()
print(f"Our flagged constituent-enzyme calls: {len(flagged)}")
print(f"Also flagged by DEEPCYPs:             {confirmed}  "
      f"({100 * confirmed / len(flagged):.0f}%)\n")
flagged.to_csv(OUT / "tableS7_deepcyps_flagged_calls.csv", index=False)
flagged

In [ ]:
# Flagged calls that DEEPCYPs does not support
disputed = flagged[~flagged["They flag too"]]
share = 1 - confirmed / len(flagged)

print(f"{len(disputed)} of our {len(flagged)} flagged calls are NOT flagged by DEEPCYPs "
      f"({100 * share:.0f}%).")
print("Two models trained on substantially the same data still disagree on "
      f"{100 * share:.0f}% of")
print("flagged calls — architecture matters even with shared training data.\n")

print("Herbs among the disputed calls:", ", ".join(sorted(disputed["Herb"].unique())), "\n")

disputed

## 9. Summary numbers, figures and tables

The numbers quoted in the web app (training counts, herb and drug counts, compounds also present in the training data,
lab-comparison counts, prior-documentation coverage) are recomputed here from the tables above and written to `outputs/data_facts.json`.
Compounds that are also in a training set (checked on standardised structure) are flagged, because their predictions are not independent tests.

In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

KEY = ["Herb", "Drug", "Enzyme"]
ENZ = ["CYP1A2", "CYP2C9", "CYP2C19", "CYP2D6", "CYP3A4"]

pairs = pd.read_csv(OUT / "table9_pair_table.csv", keep_default_na=False)
missing = pd.read_csv(OUT / "tableS1_absent_combinations.csv", keep_default_na=False)
constituents = pd.read_csv(OUT / "table4_herb_constituent_predictions.csv")
validation = pd.read_csv(OUT / "table2_model_validation_per_enzyme.csv")

import cyp_pipeline as cp
from rdkit import Chem
herb_list = pd.read_csv(PROJECT / "data" / "herb_constituents.csv")
herb_list["canonical"] = [Chem.MolToSmiles(cp.standardise(smi)) for smi in herb_list["SMILES"]]
train_sets = {e: set(pd.read_csv(PROJECT / "data" / "processed" / "all_enzymes" / f"{e}_standardised.csv")["SMILES_std"]) for e in ENZ}
in_training_by_enzyme = {e: sorted(herb_list.loc[herb_list["canonical"].isin(train_sets[e]), "Compound"].unique()) for e in ENZ}
in_training_names = sorted(set().union(*map(set, in_training_by_enzyme.values())))
in_training = dict(n_compounds=len(in_training_names), n_distinct_compounds=int(herb_list["Compound"].nunique()), names=in_training_names, per_enzyme=in_training_by_enzyme)
assert in_training["n_distinct_compounds"] == 69 and set(herb_list["Compound"]) >= set(in_training_names)

pairs["Driver in training data"] = ["Yes" if d in in_training_by_enzyme[e] else "No" for d, e in zip(pairs["Driver"], pairs["Enzyme"])]

# ---- shape of the sources
assert len(pairs) == 423 and len(missing) == 1260, (len(pairs), len(missing))
combos = pd.concat([pairs[KEY], missing[KEY]])
assert combos["Herb"].nunique() == 17 and combos["Drug"].nunique() == 97 and len(combos) == 17 * 99 and not combos.duplicated().any()

# ---- no nulls or blanks in any column the app displays
displayed = ["Herb", "Drug", "Enzyme", "Max probability", "Driver", "Assessable", "Substrate sensitivity", "Narrow therapeutic index",
             "Prior documentation", "Prior documentation basis", "Reference check", "Reference check sentence"]
for c in displayed:
    assert (pairs[c].astype(str).str.strip() != "").all(), f"blank in {c}"
assert (missing["Reason"].str.strip() != "").all()
RAN, NOT_RAN = re.compile(r"^The model assessed \d+ compounds? of this herb"), re.compile(r"^No compound of this herb could be assessed")
assert all(bool(RAN.match(x)) != bool(NOT_RAN.match(x)) for x in missing["Reason"]), "a Reason fits neither known wording"

# ---- the per-herb, per-enzyme overview (per herb and enzyme)
rows = []
for herb, g in constituents.groupby("Herb"):
    for e in ENZ:
        n = int(g[e].notna().sum())
        rows.append(dict(Herb=herb, Enzyme=e, **{"Highest probability": (round(float(g[e].max()), 6) if n else np.nan),
                                                "Assessable compounds": n, "Compounds at 0.5 or more": int((g[e] >= 0.5).sum())}))
overview = pd.DataFrame(rows)
overview["Reached 0.5"] = np.where(overview["Assessable compounds"] == 0, "Not assessed", np.where(overview["Compounds at 0.5 or more"] > 0, "Yes", "No"))
assert len(overview) == 17 * 5 and overview["Herb"].nunique() == 17
for _, r in overview.iterrows():
    sub = pairs[(pairs.Herb == r.Herb) & (pairs.Enzyme == r.Enzyme)]
    assert bool(len(sub)) == (r["Compounds at 0.5 or more"] > 0), (r.Herb, r.Enzyme)
    if len(sub):
        assert abs(float(sub["Max probability"].iloc[0]) - r["Highest probability"]) < 1e-3 and (sub["Assessable"].astype(int) == r["Assessable compounds"]).all()
    else:
        absent = missing[(missing.Herb == r.Herb) & (missing.Enzyme == r.Enzyme)]
        assert len(absent) > 0 and (bool(RAN.match(absent["Reason"].iloc[0])) == (r["Assessable compounds"] > 0))

# ---- the per-enzyme test sensitivity (cross-validation, 0.5 threshold)
sensitivity = pd.DataFrame({"Enzyme": validation["Enzyme"], "Test sensitivity": validation["CV sensitivity"].astype(float)})
assert set(sensitivity["Enzyme"]) == set(ENZ) and abs(float(sensitivity.set_index("Enzyme").loc["CYP2D6", "Test sensitivity"]) - 0.49) < 0.005

# ---- write
pairs.to_csv(OUT / "table9_pair_table.csv", index=False)
missing.to_csv(OUT / "tableS1_absent_combinations.csv", index=False)
overview.to_csv(OUT / "tableS14_herb_enzyme_overview.csv", index=False)
sensitivity.to_csv(OUT / "tableS15_enzyme_sensitivity.csv", index=False)

# ---- read back and compare with the sources, value for value
back = pd.read_csv(OUT / "table9_pair_table.csv", keep_default_na=False)
pd.testing.assert_frame_equal(back.astype(str), pairs.astype(str))
back_m = pd.read_csv(OUT / "tableS1_absent_combinations.csv", keep_default_na=False)
pd.testing.assert_frame_equal(back_m.astype(str), missing.astype(str))
back_o = pd.read_csv(OUT / "tableS14_herb_enzyme_overview.csv")
assert len(back_o) == 85 and np.allclose(back_o["Highest probability"].dropna(), overview["Highest probability"].dropna())
back_s = pd.read_csv(OUT / "tableS15_enzyme_sensitivity.csv")
assert np.allclose(back_s.set_index("Enzyme").loc[validation["Enzyme"], "Test sensitivity"].values, validation["CV sensitivity"].values)

total = sum(p.stat().st_size for p in OUT.glob("*.csv"))
print(f"PASS  wrote the pair, overview and sensitivity tables to outputs/ ({total / 1e6:.2f} MB of csv so far): {len(pairs)} pairs, {len(missing)} absent combinations, "
      f"{overview['Herb'].nunique()} herbs, {combos['Drug'].nunique()} drugs, {len(overview)} overview rows, {len(sensitivity)} enzymes; values equal the sources")
print(overview[overview["Herb"].str.startswith(("Myrrh", "Garlic"))].to_string(index=False))

In [ ]:
cleaning = pd.read_csv(OUT / "table1_training_data_cleaning.csv")
hc = pd.read_csv(DATA / "herb_constituents.csv")
subs = pd.read_csv(DATA / "cyp_drug_substrates.csv")
chembl = pd.read_csv(DATA / "chembl_measured.csv")
msk = pd.read_csv(DATA / "c3_mskcc_lookup.csv")
sel = pd.read_csv(DATA / "herb_selection_sources.csv")

# 1. training data (per enzyme: compounds kept after cleaning, share that are inhibitors)
val = validation.set_index("Enzyme")
training = {e: dict(compounds=int(val.loc[e, "Compounds"]), percent_inhibitors=float(val.loc[e, "% inhibitors"]),
                    starting_rows=int(cleaning.set_index("Enzyme").loc[e, "starting_rows"])) for e in ENZ}
assert all(training[e]["compounds"] == int(cleaning.set_index("Enzyme").loc[e, "final_rows"]) for e in ENZ)

# 2. herb compounds
assert len(hc) == len(constituents) == 71 and set(hc["Herb"]) == set(constituents["Herb"])
herb_ref = hc.groupby("Herb")["Reference"].agg(lambda x: sorted(set(x)))
assert all(len(v) == 1 for v in herb_ref), "each herb must cite exactly one review"
per_herb = {h: dict(compounds=int(len(g)), assessable=int((g["Enzymes assessable"] > 0).sum()), reference=herb_ref[h][0]) for h, g in constituents.groupby("Herb")}
tier = sel.set_index("Herb")["Evidence_tier"].str.strip()
saudi = sel.set_index("Herb")["Source"].str.contains("Saudi|Riyadh|Jeddah")
assert set(tier.index) == set(per_herb) and tier.str.contains("a").sum() == 13 and (tier == "b").sum() == 2 and (tier == "c").sum() == 2
assert saudi[tier.str.contains("a")].all()
selection = dict(n_survey_herbs=int(tier.str.contains("a").sum()), n_regulator_only_herbs=int((tier == "b").sum()), n_traditional_only_herbs=int((tier == "c").sum()))
herbs = dict(selection=selection, references=sorted(hc["Reference"].unique()), n_herbs=len(per_herb), n_compounds=int(len(hc)), n_distinct_compounds=int(hc["Compound"].nunique()), n_assessable=int(sum(v["assessable"] for v in per_herb.values())),
             n_references=int(hc["Reference"].nunique()), n_primary_quantification=int((hc["Citation_quality"].str.startswith("verified")).sum()),
             n_survey_sources=int(sel["Source"].nunique()), per_herb=per_herb)

# 3. drug list
nti_drugs = sorted(subs.loc[subs["NTI"].astype(str) == "True", "Drug"].unique())
drug_facts = dict(n_drugs=int(subs["Drug"].nunique()), n_drug_enzyme_rows=int(len(subs)), n_sensitive=int((subs["FDA_substrate_class"] == "sensitive").sum()),
                  n_moderate=int((subs["FDA_substrate_class"] == "moderate").sum()), n_nti_drugs=len(nti_drugs), nti_drug_names=nti_drugs, n_nti_pairs=int((pairs["Narrow therapeutic index"] == "Yes").sum()),
                  substrate_source=str(subs["Substrate_source"].iloc[0]), nti_source=str(subs["NTI_source"].iloc[0]))
assert herbs["n_distinct_compounds"] == in_training["n_distinct_compounds"] == 69 and herbs["n_compounds"] == 71   # 71 listings; apigenin and luteolin are listed under two herbs
assert subs["Substrate_source"].nunique() == 1 and subs["NTI_source"].nunique() == 1 and drug_facts["n_drugs"] == 97
assert drug_facts["n_sensitive"] + drug_facts["n_moderate"] == drug_facts["n_drug_enzyme_rows"] == 99

# 4. published lab measurements (ChEMBL)
reference = dict(n_measurements=int(len(chembl)), n_independent=int((chembl["independent"].astype(str) == "True").sum()),
                 n_same_source_as_training=int((chembl["independent"].astype(str) != "True").sum()),
                 n_supported=int((pairs["Reference check"] == "Supported").sum()), n_contradicted=int((pairs["Reference check"] == "Contradicted").sum()),
                 n_untested=int((pairs["Reference check"] == "Untested").sum()), n_pairs=int(len(pairs)))
assert reference["n_supported"] + reference["n_contradicted"] + reference["n_untested"] == 423
reference["n_tested"] = reference["n_pairs"] - reference["n_untested"]
assert reference["n_tested"] == reference["n_supported"] + reference["n_contradicted"]

# 5. the herb guide
in_guide = msk["In_MSKCC"] == "Yes"
guide = dict(n_herbs_checked=int(len(msk)), n_with_page=int(in_guide.sum()), n_without_page=int((~in_guide).sum()), herbs_without_page=sorted(msk.loc[~in_guide, "Herb"].str.split(" ").str[0]),
             date_checked=str(msk["Date_checked"].iloc[0]))
assert msk["Date_checked"].nunique() == 1

facts = dict(training=training, herbs=herbs, drugs=drug_facts, reference=reference, guide=guide, in_training=in_training)
pd.Series(facts).to_json(OUT / "data_facts.json", indent=2, force_ascii=False)
back = json.load(open(OUT / "data_facts.json", encoding="utf-8"))
assert back["herbs"]["n_compounds"] == 71 and back["training"]["CYP2D6"]["compounds"] == 13086 and back["guide"]["n_without_page"] == 4
print("PASS  wrote outputs/data_facts.json")
print(json.dumps({k: v for k, v in back.items() if k != "herbs"}, indent=1)[:1800]); print({k: v for k, v in back["herbs"].items() if k != "per_herb"})

## 10. Supplementary validation analyses (DOME reporting)

Added for the DOME recommendations; nothing above is changed. For each enzyme, the same cleaned compounds, the same 2,058 features
and the same scaffold-grouped 5-fold cross-validation as in section 3 are used for two models: XGBoost (the final model) and the
logistic-regression baseline of section 3.1 (standardised features, `max_iter=2000`). Metrics are computed at the 0.5 threshold.

* **Baseline for all five enzymes** (`tableS25`): fold mean and standard deviation of five metrics for both models.
* **Confidence intervals** (`tableS26`): for each metric, the mean over the 5 folds with a t-interval (`mean +/- t(0.975, 4) * sd / sqrt(5)`,
  appropriate for 5 values, though the training sets of the folds overlap so the interval is approximate), and a percentile bootstrap
  (1,000 resamples of compounds, seed 42) of the pooled out-of-fold predictions. The bootstrap treats compounds as independent,
  although compounds that share a scaffold are not.
* **Train versus held-out performance** (`tableS27`): the same metrics on the training part and on the held-out part of each fold, and the gap.
* **XGBoost versus logistic regression** (`tableS28`): paired differences over the 5 folds with a plain paired t-test and the Nadeau-Bengio
  corrected t-test. With 5 folds these tests have very little power; a non-significant result does not show that the models are equal.
* Per-fold scores of everything above: `tableS29`.

In [ ]:
import dome_analysis as dome

fold_tables, oof_predictions, oof_labels = {}, {}, {}
for enzyme in ENZYMES:
    stored = np.load(PROJECT / "data" / "processed" / "all_enzymes" / f"{enzyme}_features.npz", allow_pickle=True)
    frame = pd.read_csv(PROJECT / "data" / "processed" / "all_enzymes" / f"{enzyme}_standardised.csv")
    X_e, y_e = stored["X"], stored["y"]
    scaffolds_e = cp.scaffolds_of(frame["SMILES_std"].tolist())
    folds, oof = dome.cross_validate_both(X_e, y_e, scaffolds_e)
    fold_tables[enzyme], oof_predictions[enzyme], oof_labels[enzyme] = folds, oof, y_e
    t = folds[folds.Part == "test"]
    print(f"{enzyme:8s} test AUROC  XGBoost {t[t.Model == 'XGBoost'].AUROC.mean():.3f}   "
          f"logistic regression {t[t.Model == 'Logistic regression'].AUROC.mean():.3f}")

fold_scores = pd.concat([f.assign(Enzyme=e) for e, f in fold_tables.items()], ignore_index=True)
fold_scores = fold_scores[["Enzyme", "Model", "Fold", "Part", "n"] + dome.METRICS]
fold_scores.to_csv(OUT / "tableS29_cv_fold_scores.csv", index=False)
print("wrote outputs/tableS29_cv_fold_scores.csv")

In [ ]:
# Check: the XGBoost test scores here must equal the cross-validation of section 3 (table 2), which used the same folds and settings
table2 = pd.read_csv(OUT / "table2_model_validation_per_enzyme.csv").set_index("Enzyme")
test_scores = fold_scores[fold_scores.Part == "test"]
for enzyme in ENZYMES:
    mine = test_scores[(test_scores.Enzyme == enzyme) & (test_scores.Model == "XGBoost")]
    for metric, column in [("AUROC", "CV AUROC"), ("Sensitivity", "CV sensitivity"), ("Specificity", "CV specificity"), ("AUPRC", "CV AUPRC")]:
        assert abs(round(mine[metric].mean(), 3) - table2.loc[enzyme, column]) < 1e-9, (enzyme, metric, mine[metric].mean(), table2.loc[enzyme, column])
print("PASS  the XGBoost fold scores reproduce table 2 (AUROC, AUPRC, sensitivity, specificity) for all five enzymes")

baseline_rows = []
for enzyme in ENZYMES:
    for model in dome.MODELS:
        sel = test_scores[(test_scores.Enzyme == enzyme) & (test_scores.Model == model)]
        row = {"Enzyme": enzyme, "Model": model, "Compounds": len(oof_labels[enzyme])}
        for m in dome.METRICS:
            row[f"CV {m}"] = round(sel[m].mean(), 3)
        for m in dome.METRICS:
            row[f"CV {m} sd"] = round(sel[m].std(ddof=1), 3)
        baseline_rows.append(row)
baseline = pd.DataFrame(baseline_rows)
baseline.to_csv(OUT / "tableS25_baseline_all_enzymes.csv", index=False)
print("wrote outputs/tableS25_baseline_all_enzymes.csv\n")
baseline

In [ ]:
ci_rows = []
for enzyme in ENZYMES:
    for model in dome.MODELS:
        sel = test_scores[(test_scores.Enzyme == enzyme) & (test_scores.Model == model)]
        pooled = cp.score(oof_labels[enzyme], oof_predictions[enzyme][model])
        boot = dome.bootstrap_interval(oof_labels[enzyme], oof_predictions[enzyme][model], n_boot=1000, seed=42)
        for m in dome.METRICS:
            mean, sd, low, high = dome.t_interval(sel[m])
            ci_rows.append({"Enzyme": enzyme, "Model": model, "Metric": m,
                            "Fold mean": round(mean, 3), "Fold sd": round(sd, 3),
                            "t-interval low": round(low, 3), "t-interval high": round(high, 3),
                            "Pooled out-of-fold value": round(pooled[m], 3),
                            "Bootstrap low": round(boot[m][0], 3), "Bootstrap high": round(boot[m][1], 3)})
ci_table = pd.DataFrame(ci_rows)
ci_table.to_csv(OUT / "tableS26_cv_confidence_intervals.csv", index=False)
print("wrote outputs/tableS26_cv_confidence_intervals.csv\n")
ci_table[ci_table.Model == "XGBoost"]

In [ ]:
gap_rows = []
for enzyme in ENZYMES:
    for model in dome.MODELS:
        sel = fold_scores[(fold_scores.Enzyme == enzyme) & (fold_scores.Model == model)]
        for m in dome.METRICS:
            tr = sel[sel.Part == "train"].sort_values("Fold")[m].to_numpy()
            te = sel[sel.Part == "test"].sort_values("Fold")[m].to_numpy()
            gap_rows.append({"Enzyme": enzyme, "Model": model, "Metric": m,
                             "Train mean": round(tr.mean(), 3), "Held-out mean": round(te.mean(), 3),
                             "Gap (train - held-out)": round((tr - te).mean(), 3),
                             "Gap sd": round((tr - te).std(ddof=1), 3)})
train_vs_test = pd.DataFrame(gap_rows)
train_vs_test.to_csv(OUT / "tableS27_train_vs_test.csv", index=False)
print("wrote outputs/tableS27_train_vs_test.csv\n")
train_vs_test[train_vs_test.Metric.isin(["AUROC", "Sensitivity"])]

In [ ]:
paired_rows = []
for enzyme in ENZYMES:
    paired_rows += dome.paired_comparison(fold_tables[enzyme], enzyme)
paired = pd.DataFrame(paired_rows)
for column in paired.columns[3:]:
    if paired[column].dtype == float:
        paired[column] = paired[column].round(4)
paired.to_csv(OUT / "tableS28_xgboost_vs_logreg_paired.csv", index=False)
print("wrote outputs/tableS28_xgboost_vs_logreg_paired.csv")
print("With 5 folds the smallest possible two-sided p of a rank test is 0.0625, and the t-tests have 4 degrees of freedom: low power.\n")
paired[paired.Metric.isin(["AUROC", "Sensitivity"])]

In [ ]:
# files written by this notebook
for path in sorted(OUT.glob("*")):
    print(f"{path.name:<58}{path.stat().st_size / 1024:>9.1f} KB")